# 03. 가설 검증과 공정 범위 좁히기

**담당:** 정수진 · **Issue:** #65  
**대상:** `P1-CR1-CAL01` 소둔로 · `P1-CR1-WLD01` 용접 기록  
**분석 단위:** LOT·공정·시간 구간·센서별 관측  
**재구성일:** 2026-10-12

## 분석 목적

**분석 목적:** 02의 후보를 실제 원본·반대 사례·동시 작업으로 검증한다. 각 가설의 실제 수치·반대 기록을 보여주고, 그 근거로 알 수 있는 결론과 LOT·공정·시간창의 활용 판단을 연결한다. 수리 전후는 강종·관측 공백·종료/초기 변화·안정 참고값을 각각 대조해 수리 효과를 식별할 수 있는지도 확인한다.

네 파일의 연결: **01 시간 흐름 → 02 통계와 LOT 선정 → 03 가설 검증 → 04 모델링 준비**. 각 파일은 같은 8개 원본 CSV를 독립적으로 읽어 실행합니다. 분석 표는 노트북 출력에 남기며, 그림만 지정된 `figures/05_all_data_timeseries_jinny_03`에 저장합니다.

## 03 상세 목차 — 무엇을 보고 무엇을 확인할 것인가

| 순서 | 분석 항목 | 볼 자료와 핵심 질문 | 확인 결과와 연결 |
|---:|---|---|---|
| 1 | 가설·판정 기준 | 02 후보·관측 명제·원인 명제: 어떤 근거면 지지·기각·보류인가? | 가설별 검증 절차와 판단 한계 |
| 2 | 수리 전후 강종·신호 대조 | 강종·작업 REMARK·기록 공백 → 마지막 100분/첫 110분 TC·OP → 안정 30분 중앙값·IQR: 무엇이 바뀌고 정비 효과를 분리할 수 있는가? | 강종 교란·반복 하강/상승·안정값 차이·동일 조건 대조 부족 |
| 3 | 초기 승온 | 240061·240117 전체→초기 확대·4개 TC/OP·작업: 승온이 작업 이후 시작되는가? | 반복 승온·작업 전 선행 여부 |
| 4 | 종료 전환 | 240058 전체→후반 TC/OP/CP 원본: 함께 하강하는가? | 다중 신호 변화와 정지 명령 미확정 |
| 5 | CP 원본 전수 | 모든 CP 후보·TC-Z4·OP-Z4·CP-Z4M·냉각·정비: 공정 반응이 함께 있는가? | 피크·지속·전/중/후 효과·반대 사례 |
| 6 | 온도 후보 전수 | 같은 강종 원본·동일 상태 차이·대조 수: 실제 지속 변화인가? | 후보별 원본 검증, LOT 전체 제외 여부 |
| 7 | TC-Z9·냉각 주기 | 전체 파형·피크 간격·진폭·중심·위상: 순간 차이가 위상 때문인가? | 주기 변화와 순간 수준 차이 분리 |
| 8 | 감지·교체 | 240133 감지 전후·작업 시작·INS-006: 정비 후 효과가 있는가? | 감지 시점과 작업 전 관측의 차이 |
| 9 | 용접 전력 0 | 39 SEQ·6개 조건·양의 전력 대조: 설정 차이로 설명되는가? | 조건 일치/불일치와 원인 가설 판정 |
| 10 | 실제 동시 관측 | 온도·용접이 겹치는 4 LOT·정비: 같은 시간과 같은 소재는 같은가? | 시간 교집합 전수·소재 연결 한계 |
| 11 | 검증 근거·분석 결정 | 작업 전 TC 상승, 작업 후 CP, TC-Z9 위상 대조 등 실제 수치: 무엇을 알았고 어떤 값을 보존할 것인가? | 수치 그림 + 근거/결론/04 활용 표; 원인 확정에 필요한 자료 |

## 기본 설정

관측 시작·종료는 제공된 기록의 범위이며 소재의 실제 입·출측 시각이 아닙니다. TC의 ℃와 OP의 %는 기존 분석의 표시 가정을 유지합니다. 원본에 단위 사전·목표값·운전 모드가 없어 공학적 정상 한계는 정하지 않습니다. CP-Z4M은 정의와 단위가 확인되지 않아 원본 값을 그대로 표시합니다.

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")
FIG_DIR = os.path.join("..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

## 자료 읽기와 연결

LOT 시작과 CR1 강종변경 시각의 정확 일치를 확인하고, CAL01 설비의 WO와 부품교체를 실제 키로 연결합니다. 온도 LOT과 용접 COIL 사이의 공통 소재 키는 없으므로 시간 겹침은 동시 관측의 근거로만 사용합니다. 수리의 마지막 날짜는 하루 전체를 포함하는 계획으로 읽습니다.

In [ ]:
NOTEBOOK_NO = "03"
from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

WLD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV, "용접": WLD_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg, wld = [frames[k] for k in ["부품", "정비", "교체", "용접"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"]), (wld, ["MEAS_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG, WLD_TAG = "P1-CR1-CAL01", "P1-CR1-WLD01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
coils = wld.groupby("COIL_ID", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_SEQ=("SEQ_NO", "size"),
    PWR_MIN=("PWR", "min"), PWR_MAX=("PWR", "max"), PWR_MEDIAN=("PWR", "median"))
coils["ZERO_ALL"] = coils.PWR_MIN.eq(0) & coils.PWR_MAX.eq(0)
coils = coils.sort_values("START")
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = {240001: "#455a64", 240024: "#d55e00", 240037: "#0072b2", 240038: "#0072b2",
             240044: "#0072b2", 240058: "#a23b57", 240061: "#e69f00", 240069: "#7b2cbf",
             240091: "#009e73", 240108: "#0072b2", 240114: "#e69f00", 240117: "#7b2cbf",
             240121: "#009e73", 240133: "#7b2cbf"}
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "WLD": "#de920a", "BM": "#b52a49",
       "CBM": "#d05c76", "TBM": "#e38592", "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
DUMMY_PRIOR = {"SPHC": "사용자 후보 가정", "SPCC": "사용자 후보 가정", "SPCE": "사용자 후보 가정",
               "SPCEN": "사용자 후보 가정", "SPFH590": "사용자 제외 가정", "DP590": "사용자 제외 가정",
               "SGCC": "사용자 제외 가정", "SPCD": "추가 검토"}
lots["DUMMY_PRIOR"] = lots.GRADE.map(DUMMY_PRIOR)
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    wc = coils.loc[(coils.START <= r.END) & (coils.END >= r.START)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "WELD_COILS": len(wc), "PLAN_RELATION": " · ".join(relationships),
                         "DUMMY_PRIOR": DUMMY_PRIOR[r.GRADE]})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / f"05_all_data_timeseries_jinny_{NOTEBOOK_NO}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(text))

def table(frame, columns=None):
    shown = frame if columns is None else frame[columns]
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        display(shown)


## 그림 표시 규칙

짧은 제목, 공정과 컬럼이 보이는 소제목, 축의 대상·단위, 색 범례를 사용합니다. TC는 왼쪽, 해당 OP는 오른쪽에 놓고 CP-Z4와 CP-Z4M도 왼쪽·오른쪽으로 대조합니다. TC-Z9에는 OP가 제공되지 않아 가상의 쌍을 만들지 않습니다. 모든 그림은 저장 크기에서 글자 겹침·잘림과 범례의 그래프 침범을 검사합니다.

In [ ]:
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M 기록값\n[정의·단위 미확인]"
    if signal == "CP-Z4":
        return "CP-Z4 기록값\n[단위 미확인]"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=20, fontweight="bold", color="#17344d")
    if subtitle:
        fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=10.8, color="#526d80")
    if handles:
        fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=10.5, handlelength=2.4,
                   columnspacing=1.8, labelspacing=.65)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}") for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)

def finish(fig, name, svg=False):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다."""
    fig.set_dpi(160)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        artists += list(ax.texts)
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText]
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in fig.legends:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, facecolor="white")
    if svg:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white")
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과"})
    display(Image(filename=str(path), width=1440))
    plt.close(fig)


## 1. 가설과 판정 기준

**목적:** 검증 가능한 관측 명제와 실제 원인 명제를 구분한다. 관측 조건이 충족되면 지지, 반대 기록이 있으면 기각, 운전 코드·목표값·소재 추적·고장 라벨이 없으면 보류로 적는다. 지지는 공학적 정상·고장 확정과 다르다. 02의 같은 품질·상태·대조 규칙을 다시 계산하여 모든 온도 후보와 CP 후보를 검증한다.

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]


In [ ]:
# 같은 강종·안정 참고 상태·같은 경과 분에서 대상 LOT을 뺀 대조만 쓴다.
# 비교할 수 없는 후반 구간을 잘라내거나 대조 중앙값으로 채우지 않는다.
comparison, sensor_rows, episode_rows = {}, [], []
NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]
for grade, sub in lots.groupby("GRADE",sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    grid = pd.RangeIndex(max(len(profile[k]) for k in ids))
    for s in SIGNALS:
        values = pd.DataFrame({k:profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고")).reindex(grid) for k in ids})
        for k in ids:
            peers = values.drop(columns=k)
            n = peers.count(axis=1)
            center = peers.median(axis=1)
            delta = (values[k]-center).where(n.ge(1))
            d = pd.DataFrame({"VALUE":values[k],"CENTER":center,"N_PEERS":n,"DELTA":delta})
            comparison[k,s] = d
            v = delta.dropna()
            sensor_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"COMPARE_WINDOWS":len(v),
                                "MEDIAN_DELTA":v.median(),"Q05_DELTA":v.quantile(.05),"Q95_DELTA":v.quantile(.95),
                                "MAX_ABS_DELTA":v.abs().max(),"MIN_PEERS":int(n.loc[v.index].min()) if len(v) else 0})
            if s in NONCYCLIC:
                for a,b in runs(delta.abs().ge(3)&n.ge(2),3):
                    block = d.loc[a:b-1]
                    episode_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"START_MIN":a,"END_MIN":b,
                                         "DURATION_MIN":b-a,"MAX_ABS_DELTA_C":block.DELTA.abs().max(),
                                         "MIN_PEERS":int(block.N_PEERS.min())})
sensor_summary = pd.DataFrame(sensor_rows)
episodes = pd.DataFrame(episode_rows,columns=["LOT","GRADE","SENSOR","START_MIN","END_MIN","DURATION_MIN","MAX_ABS_DELTA_C","MIN_PEERS"])
# CP는 단위가 확인되지 않았으므로 자체 안정 참고 중앙값과 기록값 차이로만 찾는다.
cp_rows, cp_flags = [], {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = raw[k]
    ref = profile[k][("CP-Z4","median")].where(states[k].eq("안정 참고")&quality[k]["CP-Z4"].ge(.9)).median()
    hit = (g["CP-Z4"]-ref).abs().ge(.05)
    cp_flags[k] = pd.Series(profile[k].index.isin(g.loc[hit,"MINUTE"]),index=profile[k].index)
    if hit.any():
        j = (g["CP-Z4"]-ref).abs().idxmax()
        peak = g.loc[j]
        # 5초 초과 시각 공백에서 분리한다. 짧은 공백도 보간하지 않고 실제 기록 행수·최대 간격을 따로 남긴다.
        cluster = ((hit != hit.shift())|g.MEAS_DT.diff().dt.total_seconds().gt(5)).cumsum()
        pulse = g.loc[cluster.eq(cluster.loc[j])&hit]
        cp_rows.append({"LOT":k,"GRADE":r.GRADE,"BASE":ref,"PEAK_MIN":peak.ELAPSED,"PEAK_TIME":peak.MEAS_DT,
                        "PEAK_VALUE":peak["CP-Z4"],"DELTA":peak["CP-Z4"]-ref,
                        "START_MIN":pulse.ELAPSED.min(),"END_MIN":pulse.ELAPSED.max()+1/60,
                        "PULSE_WIDTH_MIN":(pulse.MEAS_DT.max()-pulse.MEAS_DT.min()).total_seconds()/60+1/60,
                        "MAX_RECORD_GAP_S":pulse.MEAS_DT.diff().dt.total_seconds().max(),
                        "RECORDED_SECONDS":len(pulse),"TOTAL_HIT_SECONDS":int(hit.sum()),
                        "TOTAL_HIT_MIN_WINDOWS":int(cp_flags[k].sum())})
cp_candidates = pd.DataFrame(cp_rows,columns=["LOT","GRADE","BASE","PEAK_MIN","PEAK_TIME","PEAK_VALUE","DELTA","START_MIN","END_MIN","PULSE_WIDTH_MIN","MAX_RECORD_GAP_S","RECORDED_SECONDS","TOTAL_HIT_SECONDS","TOTAL_HIT_MIN_WINDOWS"])


In [ ]:
hypotheses=[]
def verdict(hypothesis,result,evidence,reason,use,limit="운전 설정·인과·실제 고장 정보 부족"):
    hypotheses.append({"가설":hypothesis,"판정":result,"검증 근거":evidence,"판정 이유":reason,"04 활용":use,"남은 한계":limit})


## 2. 수리 전후 강종·신호 대조

**목적:** 수리 전후 어떤 강종이 관측됐고 어떤 작업이 있었으며 TC·OP가 어떻게 변했는지 확인한다. 형태와 안정값의 차이가 실제 정비 효과 또는 더미 사용을 입증하는지도 점검한다.

| 대조 순서 | 볼 자료 | 확인할 것 |
|---|---|---|
| 전 → 수리 → 후 | 강종·실제 시각·계획 범위·CAL01 REMARK·미관측 | 전후 강종 변경과 관측 공백, 수리 후 첫 관측의 지연 |
| 종료와 초기 확대 | 전 마지막 100분 / 후 첫 110분; TC 왼쪽·OP 오른쪽 | 여러 가열대 하강·상승과 실제 작업 착수의 선후관계 |
| 안정 참고값 | 종료 직전 안정 30분 / 승온 뒤 안정 30분; 중앙값·IQR·유효 분 | 승온·종료를 제외해도 차이가 있는지, 같은 강종 대조가 있는지 |

**가설:** 사용자 후보 강종(SPHC·SPCC·SPCE·SPCEN)과 수리 전후 위치로 실제 더미 여부를 확인할 수 있는가? 전후 강종·설정이 같지 않으면 온도 차이를 정비 효과로 해석할 수 없다. 6월 후 SPHC는 사용자 후보지만 11월 후 SPCD는 추가 검토 강종이다. 실제 역할은 용도 코드·절단/연결 실적과 소재 추적이 필요하다.

In [ ]:
dummy_rows=[]
for p in plans.itertuples():
    before=lots.loc[lots.END<p.STRT_DT].tail(1).iloc[0]
    after=lots.loc[lots.START>=p.END_EXCL].head(1).iloc[0]
    events=grade_evt.loc[grade_evt.EVT_DT.ge(p.STRT_DT)&grade_evt.EVT_DT.lt(p.END_EXCL)]
    work=cal_work.loc[cal_work.STRT_DT.ge(p.STRT_DT)&cal_work.STRT_DT.lt(p.END_EXCL)]
    dummy_rows.append({"PLAN":p.PLAN_ID,"SCOPE":p.SCOPE,"BEFORE_LOT":before.LOT_NO,"BEFORE_GRADE":before.GRADE,
                       "AFTER_LOT":after.LOT_NO,"AFTER_GRADE":after.GRADE,"AFTER_GAP_H":(after.START-p.END_EXCL).total_seconds()/3600,
                       "TEMP_ROWS_DURING":int((temp.MEAS_DT.ge(p.STRT_DT)&temp.MEAS_DT.lt(p.END_EXCL)).sum()),
                       "WELD_ROWS_DURING":int((wld.MEAS_DT.ge(p.STRT_DT)&wld.MEAS_DT.lt(p.END_EXCL)).sum()),
                       "CAL_WORK_DURING":len(work),"GRADE_EVENTS_DURING":len(events)})
    table(events[["EVT_DT","BEF_VAL","AFT_VAL"]])
    table(work[["WO_NO","DETC_DT","STRT_DT","END_DT","MNT_TYPE","REMARK"]])
    table(cal_parts.loc[cal_parts.WO_NO.isin(work.WO_NO),["WO_NO","PART_CD","PART_NM","QTY","CHG_RSN","END_DT"]])
dummy_evidence=pd.DataFrame(dummy_rows);table(dummy_evidence)
table(lots[["LOT_NO","GRADE","DUMMY_PRIOR"]])

# 관측 길이 대신 강종·운전 형태·정비 내용·관측 공백을 연결한다.
from matplotlib.patches import Rectangle
REPAIR_COLOR = {240058:"#a23b57",240061:"#d55e00",240114:"#0072b2",240117:"#7b2cbf"}
repair_pairs = []
for p, row in zip(plans.itertuples(), dummy_evidence.itertuples()):
    a, b = int(row.BEFORE_LOT), int(row.AFTER_LOT)
    pre, post = lot_meta.loc[a], lot_meta.loc[b]
    ev = grade_evt.loc[grade_evt.EVT_DT.ge(p.STRT_DT) & grade_evt.EVT_DT.lt(p.END_EXCL)]
    tasks = cal_work.loc[cal_work.STRT_DT.ge(p.STRT_DT) & cal_work.STRT_DT.lt(p.END_EXCL)]
    repair_pairs.append({"plan":p,"row":row,"before":a,"after":b,"events":ev,"work":tasks})
    note(f"**그림 목적 — {p.STRT_DT.month}월 수리 전후:** {pre.GRADE} → {post.GRADE} 변경, 수리 중 강종 이벤트·CAL01 작업 REMARK, 온도·용접 관측 공백을 함께 대조한다.")
fig=plt.figure(figsize=(24,12))
header(fig,"수리 전후 조건 대조","수리 전 마지막 관측 → 계획수리와 기록 공백 → 수리 후 첫 관측; 날짜가 다른 관측 사이를 온도 선으로 연결하지 않음",
       [Line2D([0],[0],color=REPAIR_COLOR[k],lw=3,label=f"{k} · {lot_meta.loc[k,'GRADE']}") for k in REPAIR_COLOR] +
       [Patch(facecolor="#e8eff3",label="온도·용접 미관측")],ncol=5)
for i, item in enumerate(repair_pairs):
    p,row=item["plan"],item["row"]
    a,b=item["before"],item["after"]
    pre,post=lot_meta.loc[a],lot_meta.loc[b]
    tail=state_summary.set_index("LOT").loc[a,"TAIL_START_MIN"]
    warm=state_summary.set_index("LOT").loc[b,"WARM_END_MIN"]
    top=.78-i*.35
    events="\n".join(f"{t.EVT_DT:%m/%d %H:%M}  {t.BEF_VAL} → {t.AFT_VAL}" for t in item["events"].itertuples())
    # 긴 REMARK 전수는 바로 위 표에 남기고 그림에는 대표 내용을 표시한다.
    representative="마모량 초과 · 수리기간 중 점검\n1호기 재발, SD작업 수행" if i==0 else "정기수리 실시 · 수리기간 중 점검\n1호기 재발, 정기수리 실시"
    tasks=cal_work.loc[cal_work.STRT_DT.lt(post.END)&cal_work.END_DT.gt(post.START)]
    intervals=sorted((max(t.STRT_DT,post.START),min(t.END_DT,post.END)) for t in tasks.itertuples())
    merged=[]
    for lo,hi in intervals:
        if merged and lo<=merged[-1][1]:
            merged[-1]=(merged[-1][0],max(hi,merged[-1][1]))
        else:
            merged.append((lo,hi))
    overlap=sum((hi-lo).total_seconds()/60 for lo,hi in merged)
    total=sum((hi-lo).total_seconds()/60 for lo,hi in intervals)
    work_detail=" · ".join(f"{t.MNT_TYPE} {t.REMARK} {(t.END_DT-t.STRT_DT).total_seconds()/60:.0f}분" for t in tasks.itertuples())
    work_text="실제 CAL01 작업 겹침 없음" if tasks.empty else f"{work_detail}\n공유 {total-overlap:.0f}분 → 합집합 {overlap:.0f}분"
    cards=[
        (.035,.285,REPAIR_COLOR[a],"수리 전 마지막 관측",
         f"{a} · {pre.GRADE}\n{pre.START:%m/%d %H:%M}–{pre.END:%H:%M:%S}\n전체 관측 {pre.MINUTES:.2f}분\n종료 전환 후보: 시작 후 {tail:.0f}분부터\n강종만으로 소재 용도는 확인 불가"),
        (.355,.29,"#526d80",f"{p.STRT_DT.month}월 계획수리 · {p.PLAN_ID}",
         f"{p.STRT_DT:%m/%d}–{p.END_DT:%m/%d} · {p.SCOPE}\n{events}\nCAL01 {len(item['work'])}건: 모두 TBM\n{representative}\n온도 {row.TEMP_ROWS_DURING}행 · 용접 {row.WELD_ROWS_DURING}행"),
        (.685,.28,REPAIR_COLOR[b],"수리 후 첫 관측",
         f"{b} · {post.GRADE}\n{post.START:%m/%d %H:%M}–{post.END:%H:%M:%S}\n전체 {post.MINUTES:.2f}분 · 초기 승온 0–{warm:.0f}분\n수리 마지막 날짜 종료 후 {row.AFTER_GAP_H:.2f}시간\n{work_text}")]
    for x,width,color,title,body in cards:
        fig.add_artist(Rectangle((x,top-.275),width,.285,transform=fig.transFigure,
                                 facecolor="#f7fafc" if x!=.355 else "#e8eff3",
                                 edgecolor=color,lw=1.3))
        fig.text(x+.012,top-.012,title,color=color,fontsize=14,fontweight="bold",va="top")
        fig.text(x+.012,top-.061,body,fontsize=12.6,va="top",linespacing=1.6,color="#273c4a")
    fig.text(.332,top-.135,"→",fontsize=22,color="#526d80",ha="center")
    fig.text(.662,top-.135,"→",fontsize=22,color="#526d80",ha="center")
fig.text(.035,.073,"확인: 두 수리 모두 전후 강종이 다르고, 수리 중 온도·용접 기록은 비어 있다.",fontsize=15,fontweight="bold",color="#17344d")
fig.text(.035,.035,"수리 후 첫 관측은 실제 재가동 시각과 다를 수 있다. 아래에서는 종료 하강·초기 상승과 안정 참고값을 각각 대조한다.",fontsize=12.5,color="#526d80")
finish(fig,"01_수리더미/01_수리전후_강종대조.png")

verdict("강종 가정과 수리 전후 기록으로 실제 더미 사용을 확정할 수 있다","보류",
        "6월 후 SPHC·11월 후 SPCD; 수리 중 강종변경 있음·온도/용접 0행",
        "더미 용도 코드·소재 연결·절단/연결 실적이 없음","DUMMY_PRIOR만 메타데이터; 실제 DUMMY_LABEL은 NA",
        "SPCC 온도 LOT 없음; 강종 적합성의 공학적 검토와 실제 사용 실적 필요")
note("**검증 결과:** 수리 전후 강종과 교체 WO는 확인했다. 수리 중 강종 이벤트는 생산·더미 사용 증거를 단독으로 보증하지 않는다. 강종 가정은 후보를 찾는 출발점으로 유용하지만 실제 더미 라벨은 만들 수 없다. 01의 실제 시간 흐름과 아래 승온·후반 전환을 함께 검증한다.")

### 종료 하강과 초기 상승 대조

**목적:** 전체 흐름에서 확인한 전환 구간을 확대하여 강종별 TC·OP 변화와 실제 작업의 선후관계를 읽는다. 같은 센서 범위와 같은 축 범위를 사용한다.

In [ ]:

# 전후 다른 강종을 같은 선으로 잇지 않고, 종료/초기 확대를 같은 축 범위로 대조한다.
ZONE_COLOR={3:"#0072b2",4:"#d55e00",5:"#009e73",6:"#7b2cbf"}
for item in repair_pairs:
    p=item["plan"];a,b=item["before"],item["after"]
    pre,post=lot_meta.loc[a],lot_meta.loc[b]
    tail=state_summary.set_index("LOT").loc[a,"TAIL_START_MIN"]
    warm=state_summary.set_index("LOT").loc[b,"WARM_END_MIN"]
    note(f"**그림 목적 — {p.STRT_DT.month}월 종료·초기 대조:** 수리 전 {a}의 마지막 100분과 수리 후 {b}의 첫 110분을 확대한다. 전체 관측은 3·4절의 승온·종료 전체 그림에서 보존한다. TC와 대응 OP를 좌우로 배치하고, 날짜·강종·실제 작업을 구분한다.")
    fig,axs=plt.subplots(2,2,figsize=(24,12))
    fig.subplots_adjust(left=.075,right=.97,top=.76,bottom=.24,wspace=.21,hspace=.86)
    for ri,k in enumerate([a,b]):
        meta=lot_meta.loc[k];prof=profile[k]
        start=max(0,meta.MINUTES-100) if ri==0 else 0
        end=meta.MINUTES if ri==0 else min(110,meta.MINUTES)
        ix=prof.index[(prof.index>=np.floor(start))&(prof.index<=np.floor(end))]
        x=ix.to_numpy()-meta.MINUTES if ri==0 else ix.to_numpy()
        for ci,kind in enumerate(["TC","OP"]):
            ax=axs[ri,ci]
            for zone,color in ZONE_COLOR.items():
                signal=f"{kind}-Z{zone}"
                values=prof.loc[ix,(signal,"median")].where(quality[k].loc[ix,signal].ge(.9))
                ax.plot(x,values,color=color,lw=1.65)
                low=quality[k].loc[ix,signal].lt(.9)
                if low.any():
                    ax.scatter(x[low],np.full(low.sum(),.02),transform=ax.get_xaxis_transform(),
                               color="#7c8995",marker="|",s=25,lw=.8)
            if ri==0:
                focus=tail-meta.MINUTES
                ax.axvspan(focus,0,color="#b8d5ec",alpha=.45,lw=0)
                ax.axvline(focus,color="#52738a",ls="--",lw=1.1)
                ax.set_xlim(-100,0)
            else:
                ax.axvspan(0,warm,color="#f5d98c",alpha=.35,lw=0)
                ax.axvline(warm,color="#ac7500",ls="--",lw=1.1)
                tasks=cal_work.loc[cal_work.STRT_DT.lt(meta.START+pd.Timedelta(minutes=end))&
                                   cal_work.END_DT.gt(meta.START)]
                for task in tasks.itertuples():
                    lo=max(0,(task.STRT_DT-meta.START).total_seconds()/60)
                    hi=min(end,(task.END_DT-meta.START).total_seconds()/60)
                    slot=0 if task.MNT_TYPE=="TBM" else .065
                    ax.axvspan(lo,hi,ymin=slot,ymax=slot+.055,color=COL[task.MNT_TYPE],alpha=.8,lw=0)
                    ax.axvline(lo,color=COL[task.MNT_TYPE],ls=":",lw=1.1)
                ax.set_xlim(0,end)
            ax.set_ylim(800,880) if kind=="TC" else ax.set_ylim(0,100)
            ax.set_ylabel("온도[℃]" if kind=="TC" else "제어출력[%]",labelpad=10)
            label="수리 전 마지막 100분" if ri==0 else "수리 후 첫 110분"
            ax.set_title(f"{label} · {k} {meta.GRADE} · 가열·균열 [{kind}-Z3~Z6]",loc="left",pad=12)
            style(ax,"관측 종료에 대한 상대시간[분]" if ri==0 else "관측 시작 후 경과시간[분]")
            ax.xaxis.set_major_locator(MaxNLocator(5))
            ax.yaxis.set_major_locator(MaxNLocator(5))
    gg=raw[a]
    mid=gg.ELAPSED.between(pre.MINUTES*.45,pre.MINUTES*.6)
    last=gg.ELAPSED.ge(pre.MINUTES-5)
    drop=gg.loc[last,HEAT].median()-gg.loc[mid,HEAT].median()
    rise=state_summary.set_index("LOT").loc[b]
    handles=[Line2D([0],[0],color=c,lw=2,label=f"Z{z}: TC 온도 · OP 대응 출력") for z,c in ZONE_COLOR.items()]
    handles += [Patch(facecolor="#b8d5ec",label="수리 전 종료 전환 후보"),
                Patch(facecolor="#f5d98c",label="수리 후 초기 승온 후보"),
                Patch(facecolor=COL["TBM"],label="하단 TBM 작업"),
                Patch(facecolor=COL["CBM"],label="하단 CBM 작업"),
                Line2D([0],[0],color="#7c8995",marker="|",ls="None",label="분당 유효율 90% 미만")]
    header(fig,f"{p.STRT_DT.month}월 종료·초기 대조",
           f"수리 전 {pre.START:%m/%d %H:%M}–{pre.END:%H:%M:%S} / 수리 후 {post.START:%m/%d %H:%M}–{post.END:%H:%M:%S}; 1분 중앙값, 결측 보간 없음",
           handles,ncol=5)
    fig.text(.075,.174,f"확인: 수리 전 4개 TC는 {drop.min():+.2f}~{drop.max():+.2f}℃ 하강, 수리 후에는 +{rise.RISE_MIN_C:.2f}~+{rise.RISE_MAX_C:.2f}℃ 상승.",
             fontsize=14.5,color="#17344d",fontweight="bold")
    fig.text(.075,.132,"차이 기준: 전 = 중간 45–60% 대비 마지막 5분 / 후 = 첫 5분 대비 80–100분 중앙값. OP도 함께 비교한다.",
             fontsize=11.8,color="#526d80")
    tasks=cal_work.loc[cal_work.STRT_DT.lt(post.END)&cal_work.END_DT.gt(post.START)]
    if tasks.empty:
        work_note="수리 후 관측과 실제 CAL01 작업의 겹침 0분: 상승 형태는 작업 중에만 나타나는 현상이 아니다."
    else:
        work_note="11/13 TBM 07:40–09:35: 계획 점검 수행 / CBM 08:54–12:44: 마모량 초과. 상승은 첫 작업 32분 전에 시작했다."
    fig.text(.075,.088,work_note,fontsize=12.3,color="#9a6324")
    fig.text(.075,.041,f"{pre.GRADE} → {post.GRADE}: 전후 강종이 달라 이 수준 차이만으로 정비 효과를 산출할 수 없다.",
             fontsize=12.7,color="#526d80")
    finish(fig,f"01_수리더미/02_{p.STRT_DT.month:02d}월_종료초기_대조.png")
    note(f"**그래프 결과:** 종료 하강과 초기 상승이 두 수리에서 반복된다. 전 {pre.GRADE}·후 {post.GRADE}로 비교 조건이 달라 실제 정지 명령·더미 여부·정비로 인한 개선은 이 그림으로 확정하지 않는다. 수리 후 첫 관측까지 공백 {item['row'].AFTER_GAP_H:.2f}시간도 유지한다.")


### 안정 참고 구간 대조

**목적:** 전후 각 30분의 안정 참고값을 같은 요약 규칙으로 계산한다. 강종과 운전 조건을 통제한 효과 추정이 가능한지 확인한다.

'안정 참고'는 사후 형태 분류이며 정상·완전 안정 판정이 아니다. 종료 경계 직전의 하강과 변동도 IQR·분당 표본 수로 확인한다.

In [ ]:

# 같은 규칙으로 전후 안정 참고 30분을 고르되, 서로 다른 강종임을 유지한다.
stable_rows=[]
for item in repair_pairs:
    a,b=item["before"],item["after"]
    tail=int(state_summary.set_index("LOT").loc[a,"TAIL_START_MIN"])
    warm=int(state_summary.set_index("LOT").loc[b,"WARM_END_MIN"])
    for role,k,lo,hi in [("전",a,tail-30,tail),("후",b,warm,warm+30)]:
        for s in HEAT+[f"OP-Z{i}" for i in range(3,7)]:
            mask=profile[k].index.to_series().ge(lo)&profile[k].index.to_series().lt(hi)
            mask &= quality[k][s].ge(.9)&states[k].eq("안정 참고")
            values=profile[k].loc[mask,(s,"median")].dropna()
            stable_rows.append({"PLAN":item["plan"].PLAN_ID,"전후":role,"LOT":k,
                                "강종":lot_meta.loc[k,"GRADE"],"센서":s,
                                "경과 시작[분]":lo,"끝 제외[분]":hi,"유효 분":len(values),
                                "중앙값":values.median(),"Q25":values.quantile(.25),"Q75":values.quantile(.75),
                                "동일 강종 다른 LOT 수":int(lots.GRADE.eq(lot_meta.loc[k,"GRADE"]).sum()-1)})
repair_stable=pd.DataFrame(stable_rows)
pre_tab=repair_stable.loc[repair_stable.전후.eq("전")]
post_tab=repair_stable.loc[repair_stable.전후.eq("후")]
repair_stable_delta=pre_tab.merge(post_tab,on=["PLAN","센서"],suffixes=("_전","_후"),validate="one_to_one")
repair_stable_delta["후-전"]=repair_stable_delta["중앙값_후"]-repair_stable_delta["중앙값_전"]
note("**그림 목적 — 안정 참고값 대조:** 승온·종료를 제외한 전후 각 30분에서 품질 90% 이상인 분의 중앙값과 IQR을 비교한다. 강종·유효 분·반복 LOT 수를 표로 확인해 수리 효과를 식별할 대조가 충분한지 판단한다.")
fig,axs=plt.subplots(2,2,figsize=(24,11))
fig.subplots_adjust(left=.075,right=.97,top=.76,bottom=.20,wspace=.23,hspace=.72)
for ri,item in enumerate(repair_pairs):
    a,b=item["before"],item["after"]
    for ci,kind in enumerate(["TC","OP"]):
        ax=axs[ri,ci]
        for offset,k in [(-.13,a),(.13,b)]:
            d=repair_stable.loc[repair_stable.LOT.eq(k)&repair_stable.센서.str.startswith(kind)]
            for zone in range(3,7):
                z=d.loc[d.센서.eq(f"{kind}-Z{zone}")].iloc[0]
                ax.errorbar(zone+offset,z.중앙값,
                            yerr=np.array([[z.중앙값-z.Q25],[z.Q75-z.중앙값]]),
                            color=REPAIR_COLOR[k],fmt="o",ms=8,capsize=5,lw=2)
                ax.text(zone+offset,z.중앙값+(1.2 if kind=="TC" else 2.8),
                        f"{z.중앙값:.1f}",ha="center",va="bottom",fontsize=10.5,color=REPAIR_COLOR[k])
        if kind=="TC":
            ax.set_ylim(854,866)
        else:
            data=repair_stable.loc[repair_stable.LOT.isin([a,b])&repair_stable.센서.str.startswith("OP")]
            ax.set_ylim(max(0,data.Q25.min()-8),min(100,data.Q75.max()+10))
        ax.set_xticks(range(3,7),[f"{kind}-Z{i}" for i in range(3,7)])
        ax.set_xlim(2.55,6.45)
        ax.set_ylabel("온도[℃]" if kind=="TC" else "제어출력[%]",labelpad=10)
        ax.set_title(f"{item['plan'].STRT_DT.month}월 · {lot_meta.loc[a,'GRADE']} → {lot_meta.loc[b,'GRADE']} · 가열·균열 [{kind}-Z3~Z6]",loc="left",pad=12)
        style(ax,"공정 센서[컬럼]")
        ax.yaxis.set_major_locator(MaxNLocator(4))
header(fig,"수리 전후 안정값 대조","사후 안정 참고 30분·정상 판정 아님; 점 = 분당 중앙값의 중앙값 / 세로 막대 = Q25–Q75; 품질 ≥90%",
       [Line2D([0],[0],color=REPAIR_COLOR[k],marker="o",lw=2,label=f"{k} · {lot_meta.loc[k,'GRADE']}") for k in REPAIR_COLOR],ncol=4)
windows=[]
for item in repair_pairs:
    pieces=[]
    for k in [item["before"],item["after"]]:
        d=repair_stable.loc[repair_stable.LOT.eq(k)]
        z=d.iloc[0]
        pieces.append(f"{k}: {z['경과 시작[분]']:.0f}–{z['끝 제외[분]']:.0f}분 (끝 제외), 유효 {d['유효 분'].min()}–{d['유효 분'].max()}분")
    windows.append(f"{item['plan'].STRT_DT.month}월  "+" / ".join(pieces))
fig.text(.075,.134,windows[0],fontsize=12.2,color="#526d80")
fig.text(.075,.094,windows[1],fontsize=12.2,color="#526d80")
fig.text(.075,.042,"결론: 안정값의 기술적 차이는 계산할 수 있다. DP590·SPCD 반복 LOT과 동일 운전 설정이 없어 정비 효과는 분리할 수 없다.",
         fontsize=13.8,color="#17344d",fontweight="bold")
finish(fig,"01_수리더미/03_안정참고값_대조.png")
table(repair_stable)
table(repair_stable_delta[["PLAN","센서","LOT_전","강종_전","유효 분_전","중앙값_전",
                          "LOT_후","강종_후","유효 분_후","중앙값_후","후-전"]])
note("**그래프 결과:** 전후 안정 중앙값과 IQR·유효 분을 확인했고, 그 수치 차이는 위 표에 남겼다. 두 수리 모두 강종이 다르며 전 DP590과 후 SPCD는 같은 강종의 다른 온도 LOT이 없다. 따라서 04에는 구간별 신호 특징과 정비 맥락을 넘기고 '수리 개선량' 또는 정상·고장 라벨은 만들지 않는다.")


## 3. 초기 승온의 반복과 작업 선후관계

**가설 A:** 240061·240117의 4개 가열 TC에서 초기 5분 대비 80–100분 중앙값이 20℃ 이상 상승한다.

**가설 B:** 240117의 초기 상승은 기록된 당일 작업 착수 뒤 시작한다.

**검증:** 전체 TC/OP를 먼저 보여주고 초기 110분을 원본으로 확대한다. 작업 시각과 초기 0–30분 신호 상승을 직접 대조한다.

In [ ]:
def line(ax, lot, signal, start=None, end=None, clock=False, raw_values=False, envelope=False):
    k = int(lot)
    if raw_values:
        r = lot_meta.loc[k]
        a = max(0, start or 0)
        b = min(r.MINUTES, r.MINUTES if end is None else end)
        # 표시용 1초 격자에만 재배치하여 결측 시각에서 선을 끊는다.
        times = pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)), r.START+pd.Timedelta(seconds=int(b*60)), freq="s")
        p = raw[k].set_index("MEAS_DT").reindex(times)
        x = times if clock else (times-r.START).total_seconds()/60
        y = p[signal].to_numpy()
        low = quality[k][signal].loc[quality[k].index.to_series().between(np.ceil(a),np.floor(b))].lt(.9)
        if low.any():
            marks = low.index[low].to_numpy()
            qx = r.START + pd.to_timedelta(marks,unit="min") if clock else marks
            ax.scatter(qx,np.full(len(marks),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    else:
        p = profile[k]
        p = p.loc[(p.index >= (0 if start is None else start)) & (p.index <= (np.inf if end is None else end))]
        x = lot_meta.loc[k,"START"] + pd.to_timedelta(p.index,unit="min") if clock else p.index.to_numpy()
        y = p[(signal,"median")].to_numpy()
        if envelope:
            ax.fill_between(x,p[(signal,"min")].to_numpy(),p[(signal,"max")].to_numpy(),color=LOT_COLOR[k],alpha=.14,lw=0)
        low = quality[k].loc[p.index,signal].lt(.9)
        if low.any():
            ax.scatter(np.asarray(x)[low],np.full(low.sum(),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    ax.plot(x,y,color=LOT_COLOR[k],lw=1.8)
    ax.set_title(f"{PROCESS[signal]} [{signal}]",loc="left",pad=12)
    ax.set_ylabel(unit(signal),labelpad=10)
    style(ax,"관측 시각[월/일 시:분]" if clock else "관측 시작 후 경과시간[분]")
    ax.xaxis.set_major_locator(MaxNLocator(5)) if not clock else None
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if signal == "CP-Z4M":
        ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
    if clock:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))

QUALITY_HANDLE = Line2D([0],[0],color="#7c8995",marker="|",linestyle="None",markersize=10,label="하단 눈금: 분당 유효 기록 90% 미만")
FOCUS_HANDLE = Patch(facecolor=COL["FOCUS"],alpha=.12,label="검토·확대 시간창")
ENVELOPE_HANDLE = Patch(facecolor="#97b5cc",alpha=.35,label="각 LOT 색 음영: 분당 최소–최대")

def case_plot(lot,signals,a,b,name,title,raw_values=False,whole=False,focus=None):
    grade=lot_meta.loc[lot,"GRADE"]
    ids=lots.loc[lots.GRADE.eq(grade),"LOT_NO"].astype(int).tolist()
    if not whole:
        ids=[k for k in ids if lot_meta.loc[k,"MINUTES"]>=a]
    r=lot_meta.loc[lot]
    lo=r.START if whole else r.START+pd.Timedelta(minutes=a)
    hi=r.END if whole else min(r.END,r.START+pd.Timedelta(minutes=b))
    tasks=cal_work.loc[cal_work.STRT_DT.lt(hi)&cal_work.END_DT.gt(lo)]
    detections=cal_work.loc[cal_work.DETC_DT.between(lo,hi)]
    note(f"**그림 목적 — {title}:** " + (f"전체 관측에서 {a:.1f}–{b:.1f}분의 확대 위치와 다른 동일 강종 LOT의 흐름을 확인한다." if whole else f"{a:.1f}–{b:.1f}분 원본을 동일 경과시간으로 대조하여 표시된 센서 차이와 대응 신호가 실제 기록에서도 함께 나타나는지 확인한다.") + " LOT 색은 관측별 구분이며 실제 정상·고장 판정 색이 아니다.")
    rows=int(np.ceil(len(signals)/2))
    fig,axs=plt.subplots(rows,2,figsize=(24,4+3*rows))
    fig.subplots_adjust(left=.10,right=.97,top=.70 if rows==1 else .77,bottom=.12,wspace=.28,hspace=.87)
    for ax,s in zip(np.asarray(axs).flat,signals):
        for k in ids:
            line(ax,k,s,start=None if whole else a,end=None if whole else b,raw_values=raw_values,envelope=not raw_values)
        if whole:
            ax.axvspan(a,b,color=COL["FOCUS"],alpha=.12,lw=0)
            ax.set_xlim(0,lots.loc[lots.LOT_NO.isin(ids),"MINUTES"].max()+2)
        else:
            ax.set_xlim(a,b)
            if focus is not None:
                ax.axvspan(focus[0],focus[1],color=COL["FOCUS"],alpha=.12,lw=0)
        r=lot_meta.loc[lot]
        for t in tasks.itertuples():
            x=(t.STRT_DT-r.START).total_seconds()/60;y=(t.END_DT-r.START).total_seconds()/60
            ax.axvspan(x,y,color=COL["WORK"],alpha=.12,lw=0)
        for t in detections.itertuples():
            ax.axvline((t.DETC_DT-r.START).total_seconds()/60,color=COL["DETECT"],ls=":",lw=1)
    for ax in np.asarray(axs).flat[len(signals):]:
        ax.set_axis_off()
    handles=lot_handles(ids)+[QUALITY_HANDLE]
    if not raw_values:
        handles.append(ENVELOPE_HANDLE)
    if whole or focus is not None:
        handles.append(FOCUS_HANDLE)
    if len(tasks):
        handles.append(Patch(facecolor=COL["WORK"],alpha=.2,label="대상 LOT 실제 작업"))
    if len(detections):
        handles.append(Line2D([0],[0],color=COL["DETECT"],ls=":",label="대상 LOT 감지 시각"))
    header(fig,title,f"{lot} · {grade} · {'전체→붉은 창을 다음 그림에서 확대' if whole else '원본 확대; 결측 시각에서 선을 끊음'}",
           handles,ncol=4)
    finish(fig,name)
    note(f"**그림 확인:** 대상 {lot} · 동일 강종 표시 {len(ids)} LOT · " + (f"전체 관측을 보존하고 붉은 {a:.1f}–{b:.1f}분을 다음 원본 그림에서 확인한다." if whole else f"확대 시간 {a:.1f}–{b:.1f}분. 하단 회색 눈금은 해당 센서의 분당 유효율 90% 미만이며, 원본 선은 실제 누락 시각에서 끊어진다.") + f" 대상 시간창의 실제 작업 {len(tasks)} WO·감지 {len(detections)}건이며 정량 변화와 판정 근거는 이어지는 결과에 적는다.")

def records(lot,a,b):
    r=lot_meta.loc[lot]
    lo=r.START+pd.Timedelta(minutes=a);hi=min(r.END,r.START+pd.Timedelta(minutes=b))
    return cal_work.loc[(cal_work.STRT_DT<hi)&(cal_work.END_DT>lo)|cal_work.DETC_DT.between(lo,hi)]

def local_effect(lot,signal,a,b,pad=5):
    g=raw[lot]
    def segment(lo,hi):
        v=g.loc[g.ELAPSED.ge(lo)&g.ELAPSED.lt(hi),signal].dropna()
        return v.median(),len(v),v.quantile(.95)-v.quantile(.05)
    pre,npre,vpre=segment(max(0,a-pad),a)
    mid,nmid,vmid=segment(a,b)
    post,npost,vpost=segment(b,min(lot_meta.loc[lot,"MINUTES"]+1/60,b+pad))
    return {"LOT":lot,"SENSOR":signal,"PRE_MEDIAN":pre,"DURING_MEDIAN":mid,"POST_MEDIAN":post,
            "DURING_MINUS_PRE":mid-pre,"POST_MINUS_PRE":post-pre,"PRE_N":npre,"DURING_N":nmid,"POST_N":npost,
            "DURING_Q95_Q05":vmid}


In [ ]:
warm_results=[]
for k in state_summary.loc[state_summary.WARM_END_MIN.notna(),"LOT"].astype(int):
    s=state_summary.set_index("LOT").loc[k]
    signals=[z for i in range(3,7) for z in [f"TC-Z{i}",f"OP-Z{i}"]]
    case_plot(k,signals,0,110,f"02_승온검증/{k}_01_전체.png",f"{k} 가열·출력 전체",whole=True)
    case_plot(k,signals,0,110,f"02_승온검증/{k}_02_초기원본.png",f"{k} 초기 승온 확대",raw_values=True,focus=(0,s.WARM_END_MIN))
    g=raw[k];initial=g.loc[g.ELAPSED.lt(5),HEAT].median()
    prework=g.loc[g.ELAPSED.between(25,30),HEAT].median()-initial
    rec=records(k,0,110)
    row={"LOT":k,"RISE_MIN_C":s.RISE_MIN_C,"RISE_MAX_C":s.RISE_MAX_C,"WARM_END_MIN":s.WARM_END_MIN,
         "PRE30_RISE_MIN_C":prework.min(),"FIRST_WORK_MIN":(rec.STRT_DT.min()-lot_meta.loc[k,"START"]).total_seconds()/60 if len(rec) else np.nan}
    warm_results.append(row)
    table(rec[["WO_NO","DETC_DT","STRT_DT","END_DT","MNT_TYPE","REMARK"]])
    table(pd.DataFrame([row]))
    warm_effect=[]
    for signal in signals:
        early=g.loc[g.ELAPSED.lt(5),signal].median()
        later=g.loc[g.ELAPSED.between(80,100),signal].median()
        warm_effect.append({"LOT":k,"SENSOR":signal,"FIRST5":early,"MIN80_100":later,"DELTA":later-early})
    table(pd.DataFrame(warm_effect))
    note(f"**그래프 결과:** {k} 4개 가열대 상승 {s.RISE_MIN_C:.2f}–{s.RISE_MAX_C:.2f}℃, 사후 안정 경계 후보 {s.WARM_END_MIN:.0f}분이다. 초기 25–30분도 첫 5분보다 최소 {prework.min():.2f}℃ 높다. 형태를 안정 비교에서 분리하되 실제 재가동·더미 운전이라는 원인은 보류한다.")
warm_evidence=pd.DataFrame(warm_results)
verdict("두 LOT의 4개 가열 TC에서 초기 대비 20℃ 이상 상승한다","지지",
        " · ".join(f"{r.LOT}: {r.RISE_MIN_C:.2f}–{r.RISE_MAX_C:.2f}℃" for r in warm_evidence.itertuples()),
        "원본·전체 흐름·4 TC가 반복 상승 조건을 만족","초기 승온 검토 상태를 안정 상태와 분리")
x=warm_evidence.set_index("LOT").loc[240117]
verdict("240117 초기 상승은 기록된 당일 작업 착수 뒤 시작한다","기각",
        f"첫 작업 {x.FIRST_WORK_MIN:.0f}분; 25–30분 최소 상승 {x.PRE30_RISE_MIN_C:.2f}℃",
        "기록된 작업 착수 전부터 상승이 관측됨","승온 상태와 실제 작업 플래그를 독립 보존",
        "이전 작업 영향·운전 명령·인과 효과는 별도 미확정")

<a id="termination-validation"></a>

## 4. 종료 전환 후보 전수

**가설:** 02에서 찾은 종료 전환 LOT은 가열 TC와 OP·CP가 함께 낮아지는 관측 형태다.

**검증:** 240058뿐 아니라 같은 규칙에서 찾은 240114도 전체와 후반 원본을 확인한다. 중간 45–60% 관측의 중앙값 대비 마지막 5분 차이, 신호별 변화 시작과 실제 수리·작업 위치를 계산한다. 관측 종료는 실제 라인 정지 시각이 아니다.

**01과 연결:** [6월 05B 원본](../figures/05_all_data_timeseries_jinny_01/03_06월수리/05B_수리전_종료부_원본.png)·[11월 08B 원본](../figures/05_all_data_timeseries_jinny_01/05_11월수리작업/08B_수리전_종료부_원본.png)의 날짜 축과 같은 확대 범위를 경과시간으로 읽는다. 바로 아래에 각 LOT의 전체 → 후반 TC·OP·CP 원본 → 정량 근거표가 이어진다. 240114의 같은 강종 240038은 전체 그림에서 비교할 수 있으나 기록 길이가 짧아 272–319분 후반 원본의 대조로 사용할 수 없다.

In [ ]:
tail_evidence=[]
for k in state_summary.loc[state_summary.TAIL_START_MIN.notna(),"LOT"].astype(int):
    r=lot_meta.loc[k];a=max(0,state_summary.set_index("LOT").loc[k,"TAIL_START_MIN"]-10);b=r.MINUTES
    signals=["TC-Z3","OP-Z3","TC-Z4","OP-Z4","CP-Z4","CP-Z4M"]
    case_plot(k,signals,a,b,f"03_종료검증/{k}_01_전체.png",f"{k} 종료 전환 전체",whole=True)
    case_plot(k,signals,a,b,f"03_종료검증/{k}_02_후반원본.png",f"{k} 후반 신호 확대",raw_values=True,focus=(state_summary.set_index("LOT").loc[k,"TAIL_START_MIN"],b))
    g=raw[k];mid=g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6);end=g.ELAPSED.ge(r.MINUTES-5)
    rows=[]
    for s in HEAT+[f"OP-Z{i}" for i in range(3,7)]+["CP-Z4"]:
        x,y=g.loc[mid,s].median(),g.loc[end,s].median()
        rows.append({"LOT":k,"SENSOR":s,"MIDDLE":x,"LAST5":y,"DELTA":y-x})
    d=pd.DataFrame(rows);tail_evidence.extend(rows);table(d)
    rec=cal_work.loc[cal_work.STRT_DT.between(r.END-pd.Timedelta(hours=12),r.END+pd.Timedelta(hours=36))]
    table(rec[["WO_NO","DETC_DT","STRT_DT","END_DT","MNT_TYPE","REMARK"]])
    drop=d.loc[d.SENSOR.isin(HEAT),"DELTA"]
    control=d.loc[d.SENSOR.str.startswith("OP-"),"DELTA"]
    note(f"**결과:** {k} 마지막 5분의 가열 TC 차이 {drop.min():+.2f}–{drop.max():+.2f}℃, OP 차이 {control.min():+.2f}–{control.max():+.2f}%p이다. CP 차이는 위 표의 원본 단위로 확인했다. 여러 신호의 후반 전환을 확인하지만 운전 명령·설정이 없어 실제 정지 원인은 확정하지 않는다.")
    verdict(f"{k} 종료 전 가열 TC 4개가 중간 수준보다 10℃ 이상 하강한다","지지" if drop.max()<=-10 else "기각",
            f"TC 차이 {drop.min():+.2f}–{drop.max():+.2f}℃; OP {control.min():+.2f}–{control.max():+.2f}%p",
            "원본 중간·마지막 5분 중앙값 대조","종료 전환 검토 구간 분리; 정지·더미 라벨은 NA")
tail_evidence=pd.DataFrame(tail_evidence)

## 5. 모든 CP 후보의 원본과 동시 신호

**가설 A:** CP-Z4의 큰 차이는 원본에서도 여러 기록에 존재한다.

**가설 B:** CP 변화는 항상 실제 소둔로 작업 중 발생한다.

**검증:** 02의 CP 후보 전수를 전체 → 극값 포함 펄스 확대 순서로 보여준다. 5초 이하 짧은 시각 공백이 펄스 안에 있으면 실제 폭과 초과 기록 행수를 모두 표시하고 공백을 보간하지 않는다. TC-Z4·OP-Z4·CP-Z4M·냉각의 전 5분/펄스/후 5분 중앙값·표본 수를 계산한다. CP-Z4M의 단위를 %로 바꾸거나 INS-006과 동일 센서라고 가정하지 않는다.

In [ ]:
cp_effect_rows,cp_record_rows=[],[]
for row in cp_candidates.itertuples():
    k=row.LOT;a=max(0,row.START_MIN-2);b=min(lot_meta.loc[k,"MINUTES"],row.END_MIN+2)
    case_plot(k,["CP-Z4","CP-Z4M"],a,b,f"04_CP검증/{k}_01_전체.png",f"{k} CP 전체 비교",whole=True)
    signals=["TC-Z4","OP-Z4","CP-Z4","CP-Z4M","TC-OUT1","TC-OUT2"]
    case_plot(k,signals,a,b,f"04_CP검증/{k}_02_극값원본.png",f"{k} CP·동시 신호 확대",raw_values=True,focus=(row.START_MIN,row.END_MIN))
    effects=pd.DataFrame([local_effect(k,s,row.START_MIN,row.END_MIN) for s in signals])
    cp_effect_rows.extend(effects.to_dict("records"))
    selected=effects.set_index("SENSOR")
    tc_delta=selected.loc["TC-Z4","DURING_MINUS_PRE"]
    op_delta=selected.loc["OP-Z4","DURING_MINUS_PRE"]
    cpm_delta=selected.loc["CP-Z4M","DURING_MINUS_PRE"]
    rec=records(k,row.START_MIN,row.END_MIN)
    work=rec.loc[rec.STRT_DT.lt(row.PEAK_TIME)&rec.END_DT.gt(row.PEAK_TIME)]
    cp_record_rows.append({"LOT":k,"PEAK_TIME":row.PEAK_TIME,"CP_DELTA":row.DELTA,"PULSE_WIDTH_MIN":row.PULSE_WIDTH_MIN,
                          "RECORDED_SECONDS":row.RECORDED_SECONDS,"MAX_RECORD_GAP_S":row.MAX_RECORD_GAP_S,
                          "CAL_WORK_AT_PEAK":len(work),"RECORDS_IN_PULSE":len(rec)})
    note(f"**그래프 결과:** {k} CP 극값 {row.PEAK_VALUE:.6f}, 자체 안정 참고 기준 대비 {row.DELTA:+.6f}. 극값 포함 펄스 폭 {row.PULSE_WIDTH_MIN:.2f}분·실제 초과 {row.RECORDED_SECONDS}행·최대 기록 간격 {row.MAX_RECORD_GAP_S:.0f}초다. 전체 초과 {row.TOTAL_HIT_SECONDS}행은 다른 펄스·후반 전환도 포함한다. 극값 시각의 작업은 {len(work)}건이다. 펄스 중앙값의 전 5분 대비 차이는 TC-Z4 {tc_delta:+.3f}℃·OP-Z4 {op_delta:+.3f}%p·CP-Z4M {cpm_delta:+.3e} 원본 기록값이다. 전·중·후 표본 수와 냉각도 아래 표에서 확인하며 CP만으로 센서 이상·정비 효과를 확정할 수 없다.")
    table(effects);table(rec[["WO_NO","DETC_DT","STRT_DT","END_DT","MNT_TYPE","REMARK"]])
cp_effects=pd.DataFrame(cp_effect_rows);cp_context=pd.DataFrame(cp_record_rows);table(cp_context)
verdict("CP 차이 ≥0.05는 원본 여러 기록에서 관측된다","지지",
        f"{len(cp_candidates)} LOT의 극값 포함 펄스 초과 기록 {cp_candidates.RECORDED_SECONDS.min()}–{cp_candidates.RECORDED_SECONDS.max()}행",
        "원본 피크·펄스·전체 신호와 표본 수를 전수 확인","CP 검토는 해당 분·센서에만 표시")
nonwork=cp_context.loc[cp_context.CAL_WORK_AT_PEAK.eq(0),"LOT"].tolist()
verdict("CP 변화는 항상 실제 소둔로 작업 중 발생한다","기각",
        f"작업 없는 극값 LOT: {nonwork}",
        "실제 작업과 겹치지 않는 반대 사례가 있음","CP 검토와 정비 플래그를 독립 유지",
        "작업 전후 영향·운전 설정·CP 정의·계측 로그 부족")

## 6. 온도 후보 전수 원본 검증

**가설:** 02의 분당 지속 후보는 동일 강종·상태에서 원본 초 단위로도 실제 차이가 반복된다.

**검증:** 100개 같은 규칙의 센서 후보를 모두 재계산한다. 원본을 동일 경과 초로만 맞추고 시각 누락·NaN을 채우지 않는다. 대상·대조의 분당 유효율 ≥90%와 안정 참고 상태를 유지한다. 후보 구간의 실제 비교 초 수·3℃ 초과 비율·최대 연속 기록을 남긴다. 원본 초과 비율 ≥50%는 반복 확인용 탐색 조건이며 고장 판정이 아니다.

가까운 후보는 5분 이하 간격을 포함하는 **그림 확대창**으로 묶는다. 묶은 창 전체가 이상이라는 뜻은 아니며 개별 후보의 시작·끝은 전수 표와 붉은 창에 남긴다.

In [ ]:
raw_delta,temperature_rows={},[]
def seconds_series(k,s):
    g=raw[k];ix=np.rint(g.ELAPSED.to_numpy()*60).astype(int)
    grid=pd.RangeIndex(int(np.floor(lot_meta.loc[k,"MINUTES"]*60))+1)
    y=pd.Series(g[s].to_numpy(),index=ix).reindex(grid)
    minute=grid.to_numpy()//60
    valid=states[k].reindex(minute).to_numpy()=="안정 참고"
    valid &= quality[k][s].reindex(minute).to_numpy()>=.9
    return y.where(valid)
for e in episodes.itertuples():
    key=e.LOT,e.SENSOR
    if key not in raw_delta:
        ids=lots.loc[lots.GRADE.eq(e.GRADE),"LOT_NO"].astype(int).tolist()
        target=seconds_series(e.LOT,e.SENSOR)
        peers=pd.DataFrame({k:seconds_series(k,e.SENSOR).reindex(target.index) for k in ids if k!=e.LOT})
        n=peers.count(axis=1)
        d=(target-peers.median(axis=1)).where(n.ge(2))
        raw_delta[key]=pd.DataFrame({"DELTA":d,"N_PEERS":n})
    block=raw_delta[key].loc[int(e.START_MIN*60):int(e.END_MIN*60)-1]
    v=block.DELTA.dropna();hit=block.DELTA.abs().ge(3)
    run=runs(hit);maxrun=max([b-a for a,b in run],default=0)
    ratio=hit.sum()/len(v) if len(v) else np.nan
    temperature_rows.append({"EPISODE_ID":e.Index+1,"LOT":e.LOT,"GRADE":e.GRADE,"SENSOR":e.SENSOR,
                             "START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"DURATION_MIN":e.DURATION_MIN,
                             "RAW_COMPARE_SECONDS":len(v),"RAW_EXCEED_SECONDS":int(hit.sum()),"RAW_EXCEED_RATIO":ratio,
                             "MAX_CONTIGUOUS_RECORDED_S":maxrun,"RAW_MAX_ABS_DELTA_C":v.abs().max(),
                             "VERDICT":"원본 반복 확인" if len(v)>=60 and ratio>=.5 else "비율 부족·보류"})
temperature_validation=pd.DataFrame(temperature_rows)
table(temperature_validation)
window_rows=[]
for k,sub in episodes.groupby("LOT"):
    current=None
    for e in sub.sort_values(["START_MIN","END_MIN"]).itertuples():
        if current is None:
            current={"LOT":k,"START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"SENSORS":{e.SENSOR}}
        elif e.START_MIN<=current["END_MIN"]+5:
            current["END_MIN"]=max(current["END_MIN"],e.END_MIN);current["SENSORS"].add(e.SENSOR)
        else:
            window_rows.append(current);current={"LOT":k,"START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"SENSORS":{e.SENSOR}}
    if current is not None:window_rows.append(current)
temperature_windows=pd.DataFrame(window_rows)
table(temperature_validation.groupby(["LOT","VERDICT"]).size().rename("N_EPISODES").reset_index())
note(f"**전수 결과:** {len(temperature_validation)}개 센서 후보 중 {(temperature_validation.VERDICT=='원본 반복 확인').sum()}개는 원본 비교 초의 절반 이상에서 3℃ 차이가 반복된다. 보류는 {temperature_validation.VERDICT.ne('원본 반복 확인').sum()}개다. 총 {len(temperature_windows)}개의 확대창에서 원본·해당 OP·개별 후보를 아래에 모두 보여준다.")

In [ ]:
def temperature_panels(k,a,b,sensors,name,whole=False):
    ids=lots.loc[lots.GRADE.eq(lot_meta.loc[k,"GRADE"]),"LOT_NO"].astype(int).tolist()
    if not whole:
        ids=[lot for lot in ids if lot_meta.loc[lot,"MINUTES"]>=a]
    note(f"**그림 목적 — {k} 온도 후보 {'전체' if whole else '원본 대조'}:** " + ("전체 관측 안에서 모든 개별 후보의 위치를 확인하고 다음 그림들로 확대한다." if whole else f"{a:.1f}–{b:.1f}분에서 온도 차이가 원본에도 반복되는지, 대응 OP가 어떤 흐름을 보이는지 대조한다.") + " 붉은 창은 개별 센서 후보이며 창 사이 여백까지 이상으로 간주하지 않는다.")
    ordered=[s for s in NONCYCLIC if s in sensors]
    fig=plt.figure(figsize=(24,4+3*len(ordered)))
    gs=fig.add_gridspec(len(ordered),2)
    fig.subplots_adjust(left=.10,right=.97,top=.70 if len(ordered)==1 else .77,bottom=.12,wspace=.28,hspace=.87)
    for i,s in enumerate(ordered):
        matching=s.replace("TC-","OP-")
        pair=[s,matching] if matching in OP else [s]
        for j,signal in enumerate(pair):
            ax=fig.add_subplot(gs[i,:] if len(pair)==1 else gs[i,j])
            for lot in ids:
                line(ax,lot,signal,start=None if whole else a,end=None if whole else b,raw_values=not whole,envelope=whole)
            es=episodes.loc[episodes.LOT.eq(k)&episodes.SENSOR.eq(s)]
            for e in es.itertuples():
                ax.axvspan(e.START_MIN,e.END_MIN,color=COL["FOCUS"],alpha=.14,lw=0)
            ax.set_xlim(0 if whole else a,lots.loc[lots.LOT_NO.isin(ids),"MINUTES"].max()+2 if whole else b)
    header(fig,f"{k} 온도 후보 {'전체' if whole else '원본 대조'}",
           f"{lot_meta.loc[k,'GRADE']} · {'전체 위치에서 각 붉은 창을 이어서 확대' if whole else f'{a:.1f}–{b:.1f}분 확대; 붉은 창 밖은 연결 여백'}",
           lot_handles(ids)+([ENVELOPE_HANDLE] if whole else [])+[FOCUS_HANDLE,QUALITY_HANDLE],ncol=4)
    finish(fig,name)
    if whole:
        d=episodes.loc[episodes.LOT.eq(k)]
        note(f"**그림 결과:** {k}의 {d.SENSOR.nunique()}개 센서·{len(d)}개 개별 후보가 경과 {d.START_MIN.min():.0f}–{d.END_MIN.max():.0f}분 안에 있다. 전체 관측은 유지하고 표시된 창만 아래에서 원본으로 확인한다. 흰 여백을 정상으로 확정하지 않는다.")

def verify_temperature_lot(k):
    sub=temperature_windows.loc[temperature_windows.LOT.eq(k)]
    if not len(sub):return
    sensors=set().union(*sub.SENSORS)
    temperature_panels(k,0,lot_meta.loc[k,"MINUTES"],sensors,f"05_온도전수/{k}/01_전체후보.png",whole=True)
    for j,w in enumerate(sub.itertuples(),start=2):
        a=max(0,w.START_MIN-3);b=min(lot_meta.loc[k,"MINUTES"],w.END_MIN+3)
        temperature_panels(k,a,b,w.SENSORS,f"05_온도전수/{k}/{j:02d}_원본_{w.START_MIN}_{w.END_MIN}분.png")
        d=temperature_validation.loc[temperature_validation.LOT.eq(k)&temperature_validation.START_MIN.lt(w.END_MIN)&temperature_validation.END_MIN.gt(w.START_MIN)]
        note(f"**확대 결과:** {k} {w.START_MIN}–{w.END_MIN}분의 {len(d)}개 개별 센서 후보에서 실제 비교 초·초과 비율·끊기지 않은 관측 길이를 모두 확인했다. 원본 반복 확인 {d.VERDICT.eq('원본 반복 확인').sum()}개, 보류 {d.VERDICT.ne('원본 반복 확인').sum()}개다. 선의 차이와 해당 OP를 함께 보며 전체 LOT의 고장 라벨로 확대하지 않는다.")
        z=d.loc[d.RAW_MAX_ABS_DELTA_C.idxmax()]
        note(f"**원본 수치:** 가장 큰 차이는 [{z.SENSOR}] {z.RAW_MAX_ABS_DELTA_C:.2f}℃다. 이 개별 {z.START_MIN:.0f}–{z.END_MIN:.0f}분 후보는 실제 비교 {int(z.RAW_COMPARE_SECONDS):,}초 중 {int(z.RAW_EXCEED_SECONDS):,}초({z.RAW_EXCEED_RATIO*100:.1f}%)가 3℃ 이상이다. 최대 연속 초과 기록은 {int(z.MAX_CONTIGUOUS_RECORDED_S)}초이며 결측을 채워 지속 시간을 늘리지 않았다.")
        table(d)

### 240037 온도 후보 전체 → 전수 확대

**목적:** 통계 후보의 위치를 전체에서 확인한 뒤 모든 확대창의 원본 TC와 대응 OP를 검증한다.

In [ ]:
verify_temperature_lot(240037)

### 240044 온도 후보 전체 → 전수 확대

**목적:** 통계 후보의 위치를 전체에서 확인한 뒤 모든 확대창의 원본 TC와 대응 OP를 검증한다.

In [ ]:
verify_temperature_lot(240044)

### 240061 온도 후보 전체 → 전수 확대

**목적:** 통계 후보의 위치를 전체에서 확인한 뒤 모든 확대창의 원본 TC와 대응 OP를 검증한다.

In [ ]:
verify_temperature_lot(240061)

### 240069 온도 후보 전체 → 전수 확대

**목적:** 통계 후보의 위치를 전체에서 확인한 뒤 모든 확대창의 원본 TC와 대응 OP를 검증한다.

In [ ]:
verify_temperature_lot(240069)

### 240091 온도 후보 전체 → 전수 확대

**목적:** 통계 후보의 위치를 전체에서 확인한 뒤 모든 확대창의 원본 TC와 대응 OP를 검증한다.

In [ ]:
verify_temperature_lot(240091)

### 240121 온도 후보 전체 → 전수 확대

**목적:** 통계 후보의 위치를 전체에서 확인한 뒤 모든 확대창의 원본 TC와 대응 OP를 검증한다.

In [ ]:
verify_temperature_lot(240121)

### 240133 온도 후보 전체 → 전수 확대

**목적:** 통계 후보의 위치를 전체에서 확인한 뒤 모든 확대창의 원본 TC와 대응 OP를 검증한다.

In [ ]:
verify_temperature_lot(240133)

In [ ]:
ok=int(temperature_validation.VERDICT.eq("원본 반복 확인").sum())
verdict("모든 분당 온도 지속 후보가 원본 초 단위에서도 반복 확인 조건을 만족한다",
        "지지" if ok==len(temperature_validation) else "기각",
        f"{len(temperature_validation)}개 중 원본 반복 확인 {ok}개",
        "동일 강종·상태·원본 초·대조 2 LOT 이상에서 초과 비율을 전수 대조",
        "확인된 관측 차이와 표본·비율 부족 후보를 구분; 실제 고장 라벨은 NA",
        "경과 초 정렬도 목표값·소재·속도를 완전히 통제하지 못함")

## 7. TC-Z9·냉각의 주기·위상 검증

**가설:** SPCE TC-Z9의 동일 경과 분 차이는 반복 파형의 위상 차이를 포함한다.

**검증:** TC-Z9에는 OP-Z9가 없으므로 TC-Z7·TC-Z8 전체 파형과 냉각을 비교한다. 모든 LOT의 TC-Z9·OUT1·OUT2에서 안정 참고·품질 조건의 실제 값을 사용한다. 원본 결측을 채우지 않고 유효 이웃 분 3개 이상이 있는 사후 5분 이동 중앙값에서 피크를 찾는다. 점은 원본 극값이 아닌 평활곡선의 검출 피크다. 동일한 5분 중앙 평활·prominence 5℃·최소 피크 간격(TC-Z9 80분, 냉각 30분)을 적용한다. 3개 이상 피크가 있어야 피크 간격 중앙값을 계산한다. 간격 IQR/중앙값이 20% 이하인 경우만 주기·위상 비교에 사용한다. 간격이 불일치하면 기록 공백·검출 피크 누락의 영향을 먼저 의심하고 정렬을 보류한다. 20%는 탐색 일관성 조건이며 공학적 정상 한계가 아니다. 같은 강종의 피크 정렬 뒤 차이, 중심·진폭·기간을 함께 확인한다. 아래 평활·위상는 사후 검증용이며 온라인 특징에 쓰지 않는다.

In [ ]:
cycle_rows,phase,peak_dict,cycle_values,smooth_values=[],{},{},{},{}
CYCLIC=["TC-Z9","TC-OUT1","TC-OUT2"]
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for s in CYCLIC:
        y=profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고"))
        # 원본을 채우지 않는다. 관측된 이웃 분 3개 이상의 사후 이동 중앙값으로만 피크를 찾는다.
        smooth=y.rolling(5,center=True,min_periods=3).median()
        cycle_values[k,s],smooth_values[k,s]=y,smooth
        peaks,_=find_peaks(smooth.to_numpy(),prominence=5,distance=80 if s=="TC-Z9" else 30)
        peak_dict[k,s]=peaks
        candidate_period=float(np.median(np.diff(peaks))) if len(peaks)>=3 else np.nan
        interval_iqr=float(np.quantile(np.diff(peaks),.75)-np.quantile(np.diff(peaks),.25)) if len(peaks)>=3 else np.nan
        consistent=bool(pd.notna(candidate_period) and interval_iqr/candidate_period<=.2)
        period=candidate_period if consistent else np.nan
        cycle_rows.append({"LOT":k,"GRADE":r.GRADE,"SENSOR":s,"N_PEAKS":len(peaks),"PEAK_INTERVAL_MEDIAN_MIN":candidate_period,"PERIOD_MIN":period,"PHASE_ELIGIBLE":consistent,
                          "PEAK_INTERVAL_IQR_MIN":interval_iqr,
                          "AMPLITUDE_Q95_Q05_C":y.quantile(.95)-y.quantile(.05),"CENTER_C":y.median(),
                          "ASSESSMENT":"피크 간격 일관·주기 비교 가능" if consistent else "피크 간격 불일치·보류" if len(peaks)>=3 else "완결 피크 부족·보류"})
        if pd.notna(period):
            bins=np.floor((y.index.to_numpy()-peaks[0])%period).astype(int)
            phase[k,s]=pd.DataFrame({"PHASE_MIN":bins,"VALUE":y.to_numpy()}).groupby("PHASE_MIN").VALUE.median()
cycle_table=pd.DataFrame(cycle_rows);table(cycle_table)
phase_rows=[]
for grade in lots.GRADE.drop_duplicates():
    ids=lots.loc[lots.GRADE.eq(grade),"LOT_NO"].astype(int).tolist()
    note(f"**그림 목적 — {grade} 주기·위상 대조:** 전체 TC-Z9·냉각 파형의 반복과 피크를 확인한 뒤, 충분한 피크·일관된 간격을 갖춘 LOT만 같은 피크 후 시각으로 정렬한다. 관측 시작 위상 때문에 생긴 차이와 중심·변동폭 차이를 구분한다.")
    fig,axs=plt.subplots(2,3,figsize=(24,12))
    fig.subplots_adjust(left=.09,right=.97,top=.77,bottom=.12,wspace=.28,hspace=.75)
    for j,s in enumerate(CYCLIC):
        for k in ids:
            line(axs[0,j],k,s)
            peaks=peak_dict[k,s]
            axs[0,j].scatter(peaks,smooth_values[k,s].loc[peaks],color=LOT_COLOR[k],s=22)
            if (k,s) in phase:
                p=phase[k,s]
                axs[1,j].plot(p.index,p,color=LOT_COLOR[k],lw=1.8)
        axs[1,j].set_title(f"{PROCESS[s]} [{s}] · 피크 기준 정렬",loc="left",pad=12)
        axs[1,j].set_ylabel("온도[℃]",labelpad=10);style(axs[1,j],"주기 안의 피크 후 경과시간[분]")
        valid=[k for k in ids if (k,s) in phase]
        if not valid:
            axs[1,j].set_xticks([])
            axs[1,j].set_yticks([])
            axs[1,j].text(.5,.5,"완결 피크·간격 조건 부족"+chr(10)+"위상 정렬 보류",transform=axs[1,j].transAxes,
                          ha="center",va="center",color="#526d80",fontsize=12)
        for ix,k in enumerate(valid):
            for other in valid[ix+1:]:
                pair=pd.concat([phase[k,s].rename("A"),phase[other,s].rename("B")],axis=1).dropna()
                before_pair=pd.concat([cycle_values[k,s].rename("A"),
                                       cycle_values[other,s].rename("B")],axis=1).dropna()
                before_mae=(before_pair.A-before_pair.B).abs().mean()
                phase_rows.append({"ELAPSED_MAE_C":before_mae,"GRADE":grade,"SENSOR":s,"LOT_A":k,"LOT_B":other,"N_PHASE_MIN":len(pair),
                                   "PHASE_MAE_C":(pair.A-pair.B).abs().mean(),"PHASE_MEDIAN_DELTA_C":(pair.A-pair.B).median()})
    header(fig,f"{grade} 주기·위상 대조","위: 전체 파형·검출 피크 / 아래: 피크 ≥3·간격 IQR/중앙값 ≤20%만 정렬; TC-Z9에 가상 OP를 배정하지 않음",
           lot_handles(ids)+[Line2D([0],[0],color="#34495e",marker="o",ls="None",label="각 LOT 색 점: 평활곡선 검출 피크"),QUALITY_HANDLE],ncol=4)
    finish(fig,f"06_주기검증/{grade}_전체_위상.png")
    pp=pd.DataFrame([z for z in phase_rows if z["GRADE"]==grade])
    if len(pp):
        for z in pp.itertuples():
            note(f"**위상 대조 결과:** [{z.SENSOR}] {z.LOT_A}·{z.LOT_B}의 동일 경과 분 평균 절대 차이 {z.ELAPSED_MAE_C:.3f}℃, 피크 정렬 뒤 {z.PHASE_MAE_C:.3f}℃다. 정렬 영향과 실제 중심·진폭 차이를 함께 검토한다.")
    for z in cycle_table.loc[cycle_table.GRADE.eq(grade)].itertuples():
        note(f"**파형 결과:** {z.LOT} [{z.SENSOR}] 중심 {z.CENTER_C:.3f}℃, Q95–Q05 폭 {z.AMPLITUDE_Q95_Q05_C:.3f}℃, 피크 {z.N_PEAKS}개, 주기 {f'{z.PERIOD_MIN:.1f}분' if pd.notna(z.PERIOD_MIN) else '판정 보류'}이다.")
    note(f"**결과:** {grade}의 세 센서에서 주기·변동폭·중심과 피크 정렬 차이를 각각 확인했다. 원본 관측 길이가 부족한 LOT은 정렬하지 않는다. 빈 아래 패널은 정상 판정이 아니라 피크 수·간격 일관성 부족이다. TC-Z9의 제어 관계는 TC7/8와 냉각 전체 흐름을 참고하되 별도 OP 정보가 필요하다.")
    table(cycle_table.loc[cycle_table.GRADE.eq(grade)])
    if len(pp):table(pp)
phase_validation=pd.DataFrame(phase_rows,columns=["ELAPSED_MAE_C","GRADE","SENSOR","LOT_A","LOT_B","N_PHASE_MIN","PHASE_MAE_C","PHASE_MEDIAN_DELTA_C"])
spce=phase_validation.loc[phase_validation.GRADE.eq("SPCE")&phase_validation.SENSOR.eq("TC-Z9")]
table(spce)
if len(spce):
    z=spce.iloc[0]
    verdict("SPCE TC-Z9의 동일 경과 분 차이에 반복 파형의 위상 차이가 포함된다","지지",
            f"{z.LOT_A}·{z.LOT_B} 경과 분 MAE {z.ELAPSED_MAE_C:.3f}℃ → 피크 정렬 MAE {z.PHASE_MAE_C:.3f}℃; 기간·진폭·중심 전수 표",
            "전체 반복과 정렬 후 차이가 함께 확인됨; 짧은 240133은 피크 부족",
            "순간 온도 차이와 주기·진폭·중심 변화 특징을 분리",
            "탐색 피크 규칙·운전 설정·공학적 허용 한계 미확인")
else:
    verdict("SPCE TC-Z9 위상 차이를 정량 검증할 수 있다","보류","완결 피크·위상 대조 부족","주기 정렬 조건 부족","원본 보존; 순간 차이 고장 판정 보류")

## 8. 240133 감지 전후와 교체 효과 가설

**가설 A:** 감지 시각 전후의 신호 차이를 관측할 수 있다.

**가설 B:** INS-006 교체 뒤 온도 회복 효과를 이 LOT으로 검증할 수 있다.

**검증:** 전체 → 초기 0–20분 → 감지 주변 45–70분을 보여주고 감지 전 20분·후 20분의 중앙값과 표본 수를 비교한다. 실제 작업은 관측 뒤 시작한다. 감지 시각은 이상 발생 시각이나 정답 라벨이 아니다.

In [ ]:
k=240133;r=lot_meta.loc[k]
task=cal_work.loc[cal_work.DETC_DT.between(r.START,r.END)].iloc[0]
detect=(task.DETC_DT-r.START).total_seconds()/60
signals=["TC-Z3","OP-Z3","CP-Z4","CP-Z4M","TC-OUT1","TC-OUT2"]
case_plot(k,signals,45,70,"07_감지교체/01_240133_전체.png","240133 감지 위치 비교",whole=True)
case_plot(k,signals,0,20,"07_감지교체/02_240133_초기원본.png","240133 초기 신호 확대",raw_values=True)
early=raw[k].loc[raw[k].ELAPSED.between(0,20)]
note(f"**초기 확대 결과:** 240133의 0–20분 TC-Z3 원본 범위는 {early['TC-Z3'].min():.2f}–{early['TC-Z3'].max():.2f}℃, OP-Z3는 {early['OP-Z3'].min():.2f}–{early['OP-Z3'].max():.2f}%다. 초기 변동을 감지 시점의 변화와 따로 검토하여 LOT 전체를 한 상태로 묶지 않는다.")
case_plot(k,signals,45,70,"07_감지교체/03_240133_감지원본.png","240133 감지 전후 확대",raw_values=True)
detect_rows=[]
for s in SIGNALS:
    g=raw[k]
    pre=g.loc[g.ELAPSED.ge(detect-20)&g.ELAPSED.lt(detect),s].dropna()
    post=g.loc[g.ELAPSED.ge(detect)&g.ELAPSED.lt(detect+20),s].dropna()
    detect_rows.append({"LOT":k,"SENSOR":s,"PRE20_MEDIAN":pre.median(),"POST20_MEDIAN":post.median(),
                        "DELTA":post.median()-pre.median(),"PRE_N":len(pre),"POST_N":len(post)})
detect_evidence=pd.DataFrame(detect_rows);table(detect_evidence)
table(pd.DataFrame([task])[["WO_NO","DETC_DT","STRT_DT","END_DT","MNT_TYPE","REMARK"]])
table(cal_parts.loc[cal_parts.WO_NO.eq(task.WO_NO),["WO_NO","PART_CD","PART_NM","QTY","CHG_RSN"]])
after=int(temp.MEAS_DT.ge(task.END_DT).sum())
gap=(task.STRT_DT-r.END).total_seconds()/60
z=detect_evidence.set_index("SENSOR").loc["TC-Z3","DELTA"]
note(f"**결과:** 감지 {detect:.0f}분 전후 TC-Z3 중앙값 차이 {z:+.3f}℃를 확인했다. 세 신호군의 차이는 전수 표에 있다. 주기 센서는 위상 영향을 함께 읽는다. 작업은 관측 종료 {gap:.1f}분 뒤 시작했고, 제공 온도 전체에서 이 작업 완료 후 기록은 {after}행이다.")
verdict("240133 감지 전후 신호 차이를 관측할 수 있다","지지",
        f"감지 {detect:.0f}분; TC-Z3 전후 중앙값 차이 {z:+.3f}℃; 22신호 전수 표",
        "감지 양쪽 20분의 원본 표본이 제공됨","감지 맥락 플래그와 신호 차이를 별도 보존",
        "감지는 이상 발생의 정답 시각이 아니며 전후 차이가 고장을 증명하지 않음")
verdict("INS-006 교체 후 회복 효과를 240133으로 확인할 수 있다","보류",
        f"작업 시작은 관측 종료 {gap:.1f}분 뒤; 완료 후 온도 {after}행",
        "정비 후 대조 신호가 없음","교체 사실은 WO로 보존; 효과·원인 라벨은 NA",
        "부품과 센서 매핑·교체 실측 시각·정비 후 자료 필요")

## 9. 용접 전력 0의 조건 일치 대조

**가설 A:** 두 전력 0 COIL은 39개 SEQ 전체에서 0이다.

**가설 B:** 전력 0은 제공된 6개 설정 조건 차이로 설명된다.

**검증:** SPD·LEN·SET-PWR·SET-FRQ·SET-DTY·GATE-MS를 SEQ별로 모두 맞춘다. 같은 날짜에서 일치 SEQ 수가 가장 많고 시각이 가까운 양의 전력 COIL을 우선하며, 전체 날짜에서의 최선 일치도 따로 기록한다. 조건 불일치 SEQ를 표시하고 용접기 정비·온도·강종 시각을 대조한다.

In [ ]:
COND=["SPD","LEN","SET-PWR","SET-FRQ","SET-DTY","GATE-MS"]
zero=coils.loc[coils.ZERO_ALL].sort_values("START");weld_pairs=[];weld_matches={}
positive=set(coils.loc[coils.PWR_MIN.gt(0),"COIL_ID"])
for z in zero.itertuples():
    target=wld.loc[wld.COIL_ID.eq(z.COIL_ID)].sort_values("SEQ_NO")
    joined=wld.merge(target[["SEQ_NO"]+COND],on="SEQ_NO",suffixes=("","_CASE"),validate="many_to_one")
    joined["MATCH"]=(joined[COND].to_numpy()==joined[[s+"_CASE" for s in COND]].to_numpy()).all(axis=1)
    score=joined.groupby("COIL_ID").MATCH.sum().rename("MATCH_N").reset_index().merge(coils,on="COIL_ID",validate="one_to_one")
    score=score.loc[score.COIL_ID.isin(positive)].copy()
    score["DIST_H"]=(score.START-z.START).dt.total_seconds().abs()/3600
    score["SAME_DAY"]=score.START.dt.normalize().eq(z.START.normalize())
    best=score.sort_values(["SAME_DAY","MATCH_N","DIST_H"],ascending=[False,False,True]).iloc[0]
    exact=score.sort_values(["MATCH_N","DIST_H"],ascending=[False,True]).iloc[0]
    ref=wld.loc[wld.COIL_ID.eq(best.COIL_ID)].set_index("SEQ_NO")
    t=target.set_index("SEQ_NO");matched=t[COND].eq(ref.reindex(t.index)[COND]).all(axis=1)
    weld_matches[z.COIL_ID]=(t,ref,matched)
    work=maint.loc[maint.EQP_TAG.eq(WLD_TAG)&maint.STRT_DT.lt(z.END)&maint.END_DT.gt(z.START)]
    before=maint.loc[maint.EQP_TAG.eq(WLD_TAG)&maint.END_DT.le(z.START)].sort_values("END_DT").tail(1)
    active_grade=grade_evt.loc[grade_evt.EVT_DT.le(z.START)].tail(1)
    weld_pairs.append({"CASE_COIL":z.COIL_ID,"REFERENCE_COIL":best.COIL_ID,"START":z.START,"END":z.END,"MATCH_SEQ":int(matched.sum()),
                       "MISMATCH_SEQ":", ".join(map(str,matched.index[~matched])),"REFERENCE_GAP_H":best.DIST_H,
                       "ALL_DATE_BEST":exact.COIL_ID,"ALL_DATE_MATCH_SEQ":exact.MATCH_N,"ALL_DATE_GAP_H":exact.DIST_H,
                       "WLD_WORK_OVERLAP":len(work),"TEMP_ROWS":int(temp.MEAS_DT.between(z.START,z.END).sum()),
                       "LAST_WORK_GAP_MIN":(z.START-before.END_DT.iloc[0]).total_seconds()/60 if len(before) else np.nan,
                       "ACTIVE_GRADE_CONTEXT":active_grade.AFT_VAL.iloc[0] if len(active_grade) else pd.NA})
weld_pair_table=pd.DataFrame(weld_pairs);table(weld_pair_table)
fig,axs=plt.subplots(len(zero),2,figsize=(24,11))
fig.subplots_adjust(left=.09,right=.97,top=.77,bottom=.12,wspace=.28,hspace=.72)
weld_condition_rows=[]
for i,z in enumerate(zero.itertuples()):
    t,ref,matched=weld_matches[z.COIL_ID]
    for j,s in enumerate(["PWR","SET-PWR"]):
        ax=axs[i,j]
        ax.plot(t.index,t[s],color="#d55e00",marker="o",ms=3,lw=1.8)
        ax.plot(ref.index,ref[s],color="#0072b2",marker="s",ms=3,lw=1.6)
        for seq in matched.index[~matched]:
            ax.axvspan(seq-.45,seq+.45,color=COL["FOCUS"],alpha=.15)
        ax.set_title(f"용접 [{s}] · {z.COIL_ID}",loc="left",pad=12)
        ax.set_ylabel("용접 전력 기록값[단위 미확인]" if s=="PWR" else "설정 전력[원본 단위 미확인]",labelpad=10)
        style(ax,"COIL 내부 기록 순서[SEQ_NO]")
    for s in COND:
        diff=t[s]-ref.reindex(t.index)[s]
        weld_condition_rows.append({"CASE":z.COIL_ID,"CONDITION":s,"MATCH_SEQ":int(diff.eq(0).sum()),"MIN_DELTA":diff.min(),"MAX_DELTA":diff.max()})
    mismatch=t.loc[~matched,COND].copy()
    if len(mismatch):
        table(mismatch);table(ref.reindex(mismatch.index)[COND])
header(fig,"전력 0·조건 일치 대조","왼쪽: 실측 전력 / 오른쪽: SET-PWR; 전체 6조건은 아래 전수 표; 붉은 창=조건 불일치 SEQ",
       [Line2D([0],[0],color="#d55e00",marker="o",label="전력 0 COIL"),
        Line2D([0],[0],color="#0072b2",marker="s",label="같은 날짜의 양의 전력 대조"),FOCUS_HANDLE])
finish(fig,"08_용접검증/01_전력0_조건대조.png")
table(pd.DataFrame(weld_condition_rows))
note("**결과:** 첫 사례는 동일 날짜 대조와 39/39 SEQ의 6조건이 모두 일치하고 전력만 다르다. 둘째는 38/39 일치하며 SEQ 14의 SET-PWR 차이를 표시했다. 전체 날짜 최선 39/39 대조는 시간 간격이 길어 국소 대조와 구분했다. 공통 COIL→LOT 소재 키가 없어 인접 강종 이벤트는 맥락으로만 남긴다.")
verdict("두 COIL의 39개 SEQ 전체 PWR이 0이다","지지","2 COIL × 39 SEQ의 PWR min=max=0","원본 전체 순서 확인","전력 0 관측 플래그; 원인·고장 라벨 NA")
verdict("제공된 6개 설정 조건 차이가 전력 0을 모두 설명한다","기각",
        "첫 사례는 동일 날짜 양의 전력 대조와 39/39 조건 일치",
        "제공된 설정 차이 없는 반대 사례가 있음","조건 불일치와 전력 0 기록을 분리",
        "운전 허가·장비 상태·기록 방식·실제 더미 용도 코드 없음")

## 10. 온도·용접 동시 관측 전수

**가설:** 온도와 용접의 실제 시간 교집합을 확인하면 소재 연결이나 용접의 온도 영향을 확정할 수 있다.

**검증:** 시간 겹침이 있는 4 LOT을 모두 실제 시각으로 보여준다. CAL01 작업과도 동시에 겹치는 COIL 수를 계산한다. 용접 SPD를 소둔로 이송속도나 시차 추정에 사용하지 않는다.

In [ ]:
joint_rows=[]
for k in context.loc[context.WELD_COILS.gt(0),"LOT"].astype(int):
    r=lot_meta.loc[k];g=wld.loc[wld.MEAS_DT.between(r.START,r.END)]
    wc=coils.loc[coils.START.le(r.END)&coils.END.ge(r.START)]
    tasks=cal_work.loc[cal_work.STRT_DT.lt(r.END)&cal_work.END_DT.gt(r.START)]
    triple=wc.loc[wc.apply(lambda z:any(z.START<t.END_DT and z.END>t.STRT_DT for t in tasks.itertuples()),axis=1)]
    joint_rows.append({"LOT":k,"GRADE":r.GRADE,"TEMP_START":r.START,"TEMP_END":r.END,"WELD_COILS":len(wc),
                       "WELD_ROWS":len(g),"WORK_WO":len(tasks),"TRIPLE_OVERLAP_COILS":len(triple),"SAME_MATERIAL_KEY":"없음"})
    note(f"**그림 목적 — {k} 온도·용접 동시 기록:** 동일 실제 시각에서 가열 TC·OP, CP 두 값, 냉각, 용접 PWR과 작업의 겹침을 확인한다. 소재 이동 시간이나 공통 소재 키가 없어 시각 일치만으로 공정 영향은 판정하지 않는다.")
    fig,axs=plt.subplots(3,2,figsize=(24,13))
    fig.subplots_adjust(left=.10,right=.97,top=.77,bottom=.12,wspace=.28,hspace=.87)
    for ax,s in zip(axs.flat,["TC-Z3","OP-Z3","CP-Z4","CP-Z4M","TC-OUT1"]):
        line(ax,k,s,clock=True,envelope=True)
    ax=axs[2,1];ax.scatter(g.MEAS_DT,g.PWR,color=COL["WLD"],s=5,alpha=.6)
    ax.set_title("용접 [PWR]",loc="left",pad=12);ax.set_ylabel("용접 전력 기록값[단위 미확인]",labelpad=10)
    for ax in axs.flat:
        for t in tasks.itertuples():
            ax.axvspan(t.STRT_DT,t.END_DT,color=COL["WORK"],alpha=.12)
        style(ax,"기록 시각[월/일 시:분]")
        ax.set_xlim(r.START,r.END)
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=4))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    header(fig,f"{k} 온도·용접 동시 기록",f"{r.GRADE} · 용접 {len(wc)} COIL · 실제 작업 {len(tasks)} WO · 시간 겹침은 소재 연결과 구분",
           lot_handles([k])+[Line2D([0],[0],color=COL["WLD"],marker=".",label="용접 PWR"),
                              *([Patch(facecolor=COL["WORK"],alpha=.25,label="CAL01 작업")] if len(tasks) else []),ENVELOPE_HANDLE,QUALITY_HANDLE])
    finish(fig,f"09_동시관측/{k}_온도용접_실제시각.png")
    table(pd.DataFrame([joint_rows[-1]]))
    note(f"**결과:** {k} 온도·용접 {len(wc)} COIL이 시간상 겹치며, 실제 CAL01 작업과도 겹치는 것은 {len(triple)} COIL이다. 같은 시간의 신호는 보여주지만 공정 이송 시차·소재 키·속도·목표값이 없어 용접 때문에 온도가 변했다는 설명은 보류한다.")
joint_validation=pd.DataFrame(joint_rows);table(joint_validation)
verdict("시간 겹침으로 동일 소재와 용접의 온도 영향을 확정할 수 있다","보류",
        f"4 LOT·{joint_validation.WELD_COILS.sum()} COIL 동시 관측; 240117의 삼중 겹침 확인",
        "소재 추적 키·소둔로 속도·공정 지연 정보가 없음","동시 관측 플래그와 COIL 수만 맥락으로 보존",
        "WLD SPD는 소둔로 이송속도로 사용하지 않음")

## 11. 검증 근거와 분석 결정

**목적:** 가설 판정의 개수보다 실제 근거가 분석 해석을 어떻게 바꾸는지 보여준다. 먼저 작업 전 상승·작업 후 CP·TC-Z9 위상 효과를 수치 그림으로 대조하고, 전체 가설의 근거 → 결론과 한계 → 04 활용값을 표와 그림으로 연결한다.

**확인할 것:** 신호 변화가 작업보다 먼저 있었는가? 동일 경과시간 차이를 위상이 설명하는가? 원본에서 반복된 차이를 어떤 센서·분 구간에 표시해야 하는가? 추가 자료가 없어서 만들 수 없는 라벨은 무엇인가?

추가 요청 없이 판정할 수 없는 항목은 실제 더미·정지 명령·정비 효과·고장이다. 필요한 자료와 각 자료가 해결할 질문을 아래 마지막 표에 명시한다.

In [ ]:

# 결론에 영향을 주는 세 가지 실제 수치를 그림으로 다시 확인한다.
note("**그림 목적 — 핵심 근거 대조:** 작업 전부터 상승했는가, CP 극값이 작업 안에 있는가, TC-Z9 차이가 위상 정렬 후에도 남는가를 실제 온도·시각·평균 절대 차이로 답한다.")
k=240117
g=raw[k]
initial=g.loc[g.ELAPSED.lt(5),HEAT].median()
prework=g.loc[g.ELAPSED.between(25,30),HEAT].median()-initial
late=g.loc[g.ELAPSED.between(80,100),HEAT].median()-initial
tasks=cal_work.loc[cal_work.STRT_DT.lt(lot_meta.loc[k,"END"])&cal_work.END_DT.gt(lot_meta.loc[k,"START"])].sort_values("STRT_DT")
cp=cp_candidates.loc[cp_candidates.LOT.eq(k)].iloc[0]
cbm=tasks.loc[tasks.MNT_TYPE.eq("CBM")].iloc[0]
after_cbm=(cp.PEAK_TIME-cbm.END_DT).total_seconds()/60
phase_row=phase_validation.loc[phase_validation.GRADE.eq("SPCE")&phase_validation.SENSOR.eq("TC-Z9")].iloc[0]
phase_elapsed_pair=pd.concat([cycle_values[int(phase_row.LOT_A),"TC-Z9"],
                              cycle_values[int(phase_row.LOT_B),"TC-Z9"]],axis=1).dropna()
phase_pair=pd.concat([phase[int(phase_row.LOT_A),"TC-Z9"],
                      phase[int(phase_row.LOT_B),"TC-Z9"]],axis=1).dropna()
core_work_colors={"TBM":"#825a30","CBM":"#007f86"}
fig,axs=plt.subplots(1,3,figsize=(30,13))
fig.subplots_adjust(left=.065,right=.97,top=.76,bottom=.40,wspace=.36)
ax=axs[0]
positions=np.arange(4)
ax.bar(positions-.17,prework.to_numpy(),width=.30,color="#0072b2")
ax.bar(positions+.17,late.to_numpy(),width=.30,color="#009e73")
for i,(v,w) in enumerate(zip(prework,late)):
    ax.text(i-.17,v+1,f"{v:.2f}",ha="center",va="bottom",fontsize=11,color="#0072b2")
    ax.text(i+.17,w+1,f"{w:.2f}",ha="center",va="bottom",fontsize=11,color="#009e73")
ax.set_xticks(positions,[f"TC-Z{i}" for i in range(3,7)])
ax.set_ylim(0,50)
ax.set_ylabel("초기 5분 대비 상승[℃]",labelpad=10)
ax.set_title("(a) 작업 전 상승 [TC-Z3~Z6]",loc="left",pad=14)
style(ax,"가열·균열 센서[컬럼]")
ax=axs[1]
for j,task in enumerate(tasks.itertuples()):
    ax.plot([task.STRT_DT,task.END_DT],[j,j],color=core_work_colors[task.MNT_TYPE],lw=12,solid_capstyle="butt")
ax.plot([cp.PEAK_TIME,cp.PEAK_TIME],[-.45,2.35],color="#b04482",ls=":",lw=1.1)
cp_start=lot_meta.loc[k,"START"]+pd.Timedelta(minutes=cp.START_MIN)
cp_end=lot_meta.loc[k,"START"]+pd.Timedelta(minutes=cp.END_MIN)
ax.plot([cp_start,cp_end],[2,2],color="#b04482",lw=10,solid_capstyle="butt")
ax.scatter([cp.PEAK_TIME],[2],marker="D",color="#b04482",s=60,zorder=5)
ax.set_yticks([0,1,2],["TBM 작업","CBM 작업","CP 펄스"])
for tick,color in zip(ax.get_yticklabels(),[core_work_colors["TBM"],core_work_colors["CBM"],"#b04482"]):
    tick.set_color(color)
ax.set_ylim(-.5,2.8)
ax.set_xlim(tasks.STRT_DT.min()-pd.Timedelta(minutes=15),cp.PEAK_TIME+pd.Timedelta(minutes=25))
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[8,10,12]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.set_title("(b) 작업 종료·CP 시각 [240117]",loc="left",pad=14)
style(ax,"11월 13일 기록 시각[시:분]")
ax.text(cp.PEAK_TIME,1.035,"(b)",transform=ax.get_xaxis_transform(),color="#b04482",fontsize=15,fontweight="bold",va="bottom",ha="center",clip_on=False)
ax=axs[2]
vals=[phase_row.ELAPSED_MAE_C,phase_row.PHASE_MAE_C]
ax.bar([0,1],vals,color=["#78909c","#d55e00"],width=.48)
for i,v in enumerate(vals):
    ax.text(i,v+.35,f"{v:.3f}℃",ha="center",va="bottom",fontsize=12.5)
ax.set_xticks([0,1],["동일 경과 분","피크 위상 정렬"])
ax.set_ylim(0,max(vals)*1.25)
ax.set_ylabel("두 LOT 평균 절대 차이[℃]",labelpad=10)
ax.set_title("(c) 경과 분·위상 대조 [TC-Z9]",loc="left",pad=14)
style(ax,"TC-Z9 대조 방식")
ax.yaxis.set_major_locator(MaxNLocator(4))
header(fig,"핵심 근거 대조","판정 개수 대신 원본 온도 상승·실제 작업 시각·위상 대조 수치를 제시; 원인 확정과 관측 결과는 구분",
       [Patch(facecolor="#0072b2",label="25–30분: 첫 작업 전"),
        Patch(facecolor="#009e73",label="80–100분: 초기 대비 상승"),
        Patch(facecolor=core_work_colors["TBM"],label="TBM 작업: 계획 점검 수행"),
        Patch(facecolor=core_work_colors["CBM"],label="CBM 작업: 마모량 초과"),
        Patch(facecolor="#b04482",label="CP-Z4 펄스·극값"),
        Patch(facecolor="#78909c",label="동일 경과 분 MAE"),
        Patch(facecolor="#d55e00",label="피크 위상 정렬 MAE")],ncol=4)
# 각 표시마다 근거·결론을 한 줄로 연결한다. 수치와 해석은 그대로 보존한다.
captions=[
    ("a","#0072b2",f"240117 SPCD:25–30분에 이미 최소 +{prework.min():.2f}℃ 상승 / 첫 작업은 {warm_evidence.set_index('LOT').loc[k,'FIRST_WORK_MIN']:.0f}분 뒤 | 결론: 기록된 작업 착수보다 승온이 먼저 시작됐다."),
    ("b","#b04482",f"TBM {tasks.iloc[0].STRT_DT:%H:%M}–{tasks.iloc[0].END_DT:%H:%M} / CBM {cbm.STRT_DT:%H:%M}–{cbm.END_DT:%H:%M} / CP 극값 {cp.PEAK_TIME:%H:%M:%S}, CBM 완료 {after_cbm:.2f}분 뒤 | 결론: CP 검토와 작업 플래그를 분리한다."),
    ("c","#d55e00",f"SPCE {int(phase_row.LOT_A)}·{int(phase_row.LOT_B)}:MAE {phase_row.ELAPSED_MAE_C:.3f}℃({len(phase_elapsed_pair)}쌍) → 위상 정렬 {phase_row.PHASE_MAE_C:.3f}℃({len(phase_pair)}쌍) | 결론: 주기·진폭·중심·위상을 함께 비교한다.")]
for i,(label,color,text) in enumerate(captions):
    fig.text(.065,.28-i*.075,f"({label}) {text}",fontsize=15,color=color,va="top")
for ax in axs:
    ax.set_title(ax.get_title(loc="left"),loc="left",fontsize=18,pad=43)
    ax.tick_params(labelsize=13);ax.xaxis.label.set_fontsize(14);ax.yaxis.label.set_fontsize(14)
fig._suptitle.set_fontsize(26)
for text in fig.legends[0].get_texts():text.set_fontsize(13)
fig.legends[0].set_bbox_to_anchor((.031,.90))
fig.text(.065,.037,"남은 한계: 작업 이전 영향·운전 명령·CP 계측 정의·정상 허용 기준은 확인되지 않았다. 이 수치를 실제 고장 라벨로 바꾸지 않는다.",
         fontsize=11.9,color="#526d80")
finish(fig,"10_검증결론/02_핵심근거_대조.png")
note(f"**그래프 결과:** 240117은 첫 작업 전에 최소 {prework.min():.2f}℃ 상승했고 CP 극값은 CBM 완료 {after_cbm:.2f}분 뒤였다. SPCE TC-Z9의 평균 절대 차이는 {phase_row.ELAPSED_MAE_C:.3f}℃에서 {phase_row.PHASE_MAE_C:.3f}℃로 줄었다. 작업 선후관계와 위상은 해석을 바꾸는 근거지만 인과 효과나 실제 고장을 확정하지 않는다.")


### 검증 근거 → 결론 → 04 활용

**목적:** 수치 그림과 앞 절의 원본 검증을 바탕으로 14개 가설의 핵심 근거를 9개 분석 질문으로 정리한다. 무엇이 확인됐고 어떤 비교가 더 필요한지, 어떤 값을 남길지 연결한다. 판정별 개수 막대는 사용하지 않는다.

In [ ]:

# 판정은 원래 상세 표에 보존하고, 그림은 근거 → 결론 → 04 입력의 연결을 보여준다.
hypothesis_results=pd.DataFrame(hypotheses)
table(hypothesis_results)
note("**그림 목적 — 검증 근거와 분석 결정:** 14개 가설을 9개 분석 질문으로 묶어 실제 숫자·반대 사례, 확인한 결론과 남은 한계, 04에 보존할 값을 연결한다. 상세 판정은 위 표의 근거와 함께 읽는다.")
heat_tail=tail_evidence.loc[tail_evidence.SENSOR.isin(HEAT)]
detect_delta=detect_evidence.set_index("SENSOR").loc["TC-Z3","DELTA"]
zero_coils=int(coils.ZERO_ALL.sum())
detect_task=cal_work.loc[cal_work.DETC_DT.between(lot_meta.loc[240133,"START"],lot_meta.loc[240133,"END"])].iloc[0]
detect_work_gap=(detect_task.STRT_DT-lot_meta.loc[240133,"END"]).total_seconds()/60
detect_after_rows=int(temp.MEAS_DT.ge(detect_task.END_DT).sum())
weld_matches_text=" · ".join(f"{n}/39" for n in weld_pair_table.MATCH_SEQ)
j117=joint_validation.set_index("LOT").loc[240117]
time_coils=int(context.loc[context.WELD_COILS.gt(0),"WELD_COILS"].sum())
matched_temp=int(temperature_validation.VERDICT.eq("원본 반복 확인").sum())
z117=warm_evidence.set_index("LOT").loc[240117]
samegrade_delta=repair_stable_delta.loc[repair_stable_delta.센서.isin(HEAT)]
for item in repair_pairs:
    plan_id=item["plan"].PLAN_ID
    d=samegrade_delta.loc[samegrade_delta.PLAN.eq(plan_id),"후-전"]
    item["stable_tc_delta"]=(d.min(),d.max())
d6=repair_pairs[0]["stable_tc_delta"];d11=repair_pairs[1]["stable_tc_delta"]
evidence_rows=[
    ["수리 전후·더미",
     f"DP590 → SPHC / SPCEN → SPCD\n안정 TC 후-전: 6월 {d6[0]:+.2f}~{d6[1]:+.2f}℃\n11월 {d11[0]:+.2f}~{d11[1]:+.2f}℃; 수리 중 온도·용접 0행",
     "전후 강종이 달라 효과를 분리할 수 없음\n강종 적합성 가정과 실제 더미 사용은 다름\n소재 용도·설정·연결 기록 추가 확인",
     "강종 가정·수리 맥락 보존\n더미 라벨·정비 효과 값은 NA"],
    ["초기 승온·작업",
     f"061·117의 4개 TC 모두 +{warm_evidence.RISE_MIN_C.min():.2f}℃ 이상\n117: 작업 착수 {z117.FIRST_WORK_MIN:.0f}분, 그 전 25–30분\n이미 최소 +{z117.PRE30_RISE_MIN_C:.2f}℃",
     "상승은 기록된 작업 착수 전에 시작\n'이 작업이 승온을 시작'은 반대 기록 있음\n승온 원인·실제 재가동 명령은 미확정",
     "초기 승온 형태·실제 작업 독립 보존\n안정 참고 비교에서 승온 구간 분리"],
    ["종료 전환",
     f"058·114: 4개 TC 마지막 5분 하강\n중간 대비 {heat_tail.DELTA.min():+.2f}~{heat_tail.DELTA.max():+.2f}℃\nOP·CP 후반 변화도 원본 확인",
     "다중 신호 종료 전환 형태가 반복됨\n실제 정지·절단·더미 명령은 확인 불가",
     "종료 전환 구간 표시\n관측 종료와 실제 라인 정지를 구분"],
    ["CP·정비",
     f"{len(cp_context)} LOT CP 극값 시각의 CAL01 작업 0건\n117 CP: {cp.PEAK_TIME:%H:%M:%S}\nCBM 종료 {after_cbm:.1f}분 뒤",
     "'CP가 항상 작업 중 발생'은 맞지 않음\n작업 전후 영향까지 없다는 의미는 아님\nTC-Z4·OP-Z4·CP-Z4M 함께 확인",
     "CP 펄스·작업 맥락 독립 보존\nCP 변화만으로 센서/부품 고장 라벨 금지"],
    ["동일 강종 온도",
     f"{matched_temp}/{len(temperature_validation)}개 센서·분 구간 후보\n원본 초 단위 반복 확인 조건 충족\n강종·상태·동일 경과시간 대조",
     "분당 요약에만 생긴 차이는 아님\n반복된 온도 차이와 실제 고장은 다름\n대응 OP·품질·다른 구간을 추가 대조",
     "해당 분·센서의 검토 플래그\nLOT 전체를 고장·정상으로 확정하지 않음"],
    ["TC-Z9·주기",
     f"SPCE 037·121 [TC-Z9]\n경과 분 MAE {phase_row.ELAPSED_MAE_C:.3f}℃\n피크 정렬 MAE {phase_row.PHASE_MAE_C:.3f}℃",
     "순간 온도 차이에 위상 영향이 포함됨\n주기·진폭·중심 변화와 구분해야 함\nTC-Z9 제어 관계는 추가 OP 자료 필요",
     "주기·진폭·중심 특징 분리\n완결 피크 부족은 비교 보류"],
    ["감지·부품 효과",
     f"133 감지 전후 TC-Z3 {detect_delta:+.3f}℃\n작업은 관측 종료 {detect_work_gap:.1f}분 뒤 착수\n이 작업 완료 후 제공 온도 {detect_after_rows}행",
     "감지 양쪽의 신호 차이는 확인 가능\n정비 후 신호가 없어 회복 효과 검증 불가\n감지 시각은 고장 발생 정답이 아님",
     "감지·WO·부품 교체 사실 보존\nINS-006 효과·원인 라벨은 NA"],
    ["용접 전력 0",
     f"{zero_coils} COIL 각각 39/39 SEQ 전력 0\n동일 날짜 양의 전력 대조: 설정 6조건\nSEQ 일치율 {weld_matches_text}",
     "제공 설정 차이만으로 0을 설명 못함\n운전 허가·전원·PWR 기록 방식 추가 확인\n소둔 온도에 미친 영향은 미확정",
     "전력 0 검토·조건 일치율 보존\n전력 0을 용접기 고장 라벨로 확정하지 않음"],
    ["온도·용접·작업",
     f"4 LOT에 {time_coils} COIL 시간 교집합\n117의 {j117.WELD_COILS} COIL 중 {j117.TRIPLE_OVERLAP_COILS}은 작업과도 겹침\n온도 LOT ↔ 용접 COIL 소재 키 없음",
     "동시에 기록됐다는 사실을 확인\n같은 소재·정비 효과·원인 증거는 부족\n소재 추적과 공정 지연을 추가 확인",
     "시간 교집합 맥락만 보존\nCOIL을 LOT에 소재 키로 결합하지 않음"]]
evidence_decisions=pd.DataFrame(evidence_rows,columns=["검증 질문","실제 근거","알게 된 것·남은 한계","04 분석 값"])
table(evidence_decisions)
fig=plt.figure(figsize=(25,15))
header(fig,"검증 근거·분석 결정","각 행을 왼쪽부터 읽으면 어떤 기록이 어떤 결론과 모델링 준비 값으로 이어지는지 확인할 수 있음",
       [Patch(facecolor="#e5f1fa",label="실제 기록·수치"),
        Patch(facecolor="#e6f4ed",label="결론·남은 한계"),
        Patch(facecolor="#f1eafa",label="04에 전달할 값")],ncol=3)
xs=[.035,.195,.475,.755]
widths=[.16,.28,.28,.21]
headings=["검증 질문","실제 근거","그래서 알게 된 것","04에 전달할 값"]
top=.79;row_height=.075
for x,width,title in zip(xs,widths,headings):
    fig.add_artist(Rectangle((x,top),width,.038,transform=fig.transFigure,facecolor="#17344d",edgecolor="white"))
    fig.text(x+.009,top+.010,title,color="white",fontsize=13.8,fontweight="bold",va="bottom")
for i,row in enumerate(evidence_rows):
    y=top-(i+1)*row_height
    for j,(x,width,value) in enumerate(zip(xs,widths,row)):
        bg="#f7fafc" if j==0 else ["#e5f1fa","#e6f4ed","#f1eafa"][j-1]
        fig.add_artist(Rectangle((x,y),width,row_height,transform=fig.transFigure,
                                 facecolor=bg,edgecolor="white",lw=1.2))
        fig.text(x+.009,y+row_height-.014,value,va="top",fontsize=12.6 if j else 13.5,
                 color="#17344d" if j==0 else "#273c4a",
                 fontweight="bold" if j==0 else "normal",linespacing=1.55)
fig.text(.035,.058,"결론: 관측 형태·센서 차이·정비 맥락은 모델링 준비에 남길 수 있다. 더미·실제 고장·정비 효과는 추가 정답 기록이 필요하다.",
         fontsize=14,color="#17344d",fontweight="bold")
fig.text(.035,.028,"위 표의 수치 조건과 전수 원본 검증은 앞 절의 그림·상세 표에서 확인한다. 판단 개수는 결론의 근거로 사용하지 않는다.",
         fontsize=11.8,color="#526d80")
finish(fig,"10_검증결론/01_가설_판정.png")
note("**그래프 결과:** 승온·종료 형태, 원본에 반복된 온도 차이, CP 펄스, 주기·위상, 감지와 동시 기록을 각각 활용할 수 있다. 작업 전에 이미 상승했고 CP 극값은 작업과 겹치지 않았으므로 신호 검토와 정비 플래그를 독립 유지한다. 강종 교란·정비 후 미관측·소재 키 부재 때문에 실제 더미·고장·수리 효과의 라벨은 만들 수 없다.")

remaining=pd.DataFrame([
    {"추가 자료":"더미 용도·절단/연결·COIL↔LOT 소재 추적","해결할 판단":"더미 실사용·수리 전후 소재 역할"},
    {"추가 자료":"CAL01 운전 모드·정지/재가동·목표 온도·라인속도·두께","해결할 판단":"상태 전환 원인·조건 통제·공정 지연"},
    {"추가 자료":"CP-Z4/CP-Z4M 계측 정의·단위·설비/부품 센서 매핑","해결할 판단":"CP 이상 의미·INS-006 연관"},
    {"추가 자료":"정비 후 동일 조건 온도·실제 고장 발생/확인 기록","해결할 판단":"정비 효과·고장 예측 평가·정상 검증"},
    {"추가 자료":"용접기 운전 허가·전원·PWR 기록 방식","해결할 판단":"전력 0 원인"}])
table(remaining)
note("**04로 이어지는 값:** 사후 운전 형태, 실제 작업·감지 맥락, 센서별 품질, 원본 온도 차이 검토, CP 펄스 검토, 주기 비교 가능 여부를 각각 남긴다. 이 표와 규칙은 04에서 같은 원본으로 재계산하며 온라인 입력과 사후 해석용 값을 분리한다.")
table(pd.DataFrame(figure_checks))
note(f"**시각화 확인:** {len(figure_checks)}개 그림의 글자 겹침·잘림·범례 위치 검사를 통과했다.")